# Trabajo Práctico Integrador – Introducción al Análisis de Datos## Notebook maestro consolidado: Entregas 1 a 6**Estudiante:** Aguillón, Fernando  **Comisión:** 5  **Dataset:** E – Reservas hoteleras  **Caso:** análisis y predicción de cancelaciones en reservas hoteleras### Organización del documentoEste notebook reúne, en orden, el contenido del maestro original (entregas 1 y 2) y los notebooks individuales de las entregas 3, 4, 5 y 6. Se conservan las explicaciones, procedimientos, consultas, visualizaciones, modelos y evaluaciones de cada entrega.### Requisito previo: archivo de datosAntes de ejecutar las celdas, colocá el archivo **`hotel booking TPI grupo E.csv`** en el directorio de trabajo del notebook. En Google Colab, subilo al panel de archivos de la sesión (normalmente queda en `/content/`). El archivo de datos no está almacenado en este repositorio.Ejecutá las celdas en orden, desde el principio. Las entregas posteriores vuelven a cargar el CSV y reconstruyen sus propios objetos para no depender de variables temporales creadas en etapas anteriores.### DependenciasLos notebooks instalan o comprueban los paquetes de R que necesitan en cada etapa. Se utilizan, entre otros, `tidyverse`, `lubridate`, `ggplot2`, `GGally`, `DBI`, `duckdb`, `scales` y `rpart`.**Entorno:** kernel de R. En Colab, seleccioná un entorno con R; en una instalación local, abrí el archivo con Jupyter y un kernel de R.

In [ ]:
# Comprobación inicial del entorno y del dataset compartidoarchivo_dataset <- "hotel booking TPI grupo E.csv"candidatos_dataset <- c(  file.path("/content", archivo_dataset),  file.path(getwd(), archivo_dataset),  archivo_dataset)rutas_validas <- candidatos_dataset[file.exists(candidatos_dataset)]if (length(rutas_validas) == 0) {  stop(    paste0(      "No se encontró el archivo '", archivo_dataset, "'. ",      "En Google Colab, subilo al panel de archivos (directorio /content/) ",      "y volvé a ejecutar esta celda."    )  )}ruta_csv_compartida <- normalizePath(rutas_validas[1], mustWork = TRUE)cat("Dataset localizado en:", ruta_csv_compartida, "\\n")cat("Directorio de trabajo:", getwd(), "\\n")

---# ENTREGA 1 Y 2 — Limpieza, preparación y comprensión inicial de los datosContenido incorporado desde el notebook original **TPI_Analisis_De_Datos_Maestro.ipynb**.

# Trabajo Práctico Integrador - Introducción al Análisis de Datos**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.> TPI - Maestro

##Datos de identificación- **Apellido y nombre:** Aguillón, Fernando- **Comisión:** 5- **Dataset asignado:** E- **Entrega:** 1- **Fecha:** 01/09/2026

##Importación de libreríasImportar solamente las librerías necesarias para la entrega que se está desarrollando.

In [ ]:
paquetes <- c("tidyverse", "lubridate")instalar <- paquetes[!paquetes %in% rownames(installed.packages())]if (length(instalar) > 0) {  install.packages(instalar)}library(tidyverse)library(lubridate)

##Carga del datasetCargar el archivo CSV correspondiente a la comisión. El notebook debe funcionar desde el dataset original, no desde objetos creados en otros notebooks.

In [ ]:
datos <- read.csv(ruta_csv_compartida, stringsAsFactors = FALSE)

##Presentación del problemaEl presente trabajo plantea el análisis de un conjunto de datos correspondiente a reservas realizadas en hoteles urbanos y hoteles tipo resort. El objetivo principal es identificar patrones y características que puedan estar relacionados con la cancelación de las reservas.El dataset contiene información sobre las reservas, incluyendo fechas de llegada, duración de las estadías, cantidad de huéspedes, canales de distribución, tipos de clientes, historial de cancelaciones y tarifas promedio por noche, entre otras variables.A partir de esta información, se busca comprender el comportamiento de las reservas y estudiar qué características podrían estar asociadas con una mayor probabilidad de cancelación.La variable principal del estudio será `is_canceled`, que indica si una reserva fue cancelada o no:- `0`: reserva no cancelada.- `1`: reserva cancelada.El análisis se desarrollará mediante distintas etapas de preparación, exploración, análisis, visualización e interpretación de datos. En etapas posteriores también se buscará evaluar la posibilidad de construir un modelo predictivo que permita estimar la probabilidad de cancelación de una reserva.

## Relación entre datos, información y conocimientoPara comprender el propósito del análisis, es importante diferenciar los conceptos de datos, información y conocimiento, ya que representan distintos niveles de aprovechamiento de los elementos disponibles.**Datos:** son los valores individuales que se encuentran almacenados en el dataset, sin que necesariamente tengan una interpretación por sí solos. Por ejemplo, un registro puede contener un valor `1` en la variable `is_canceled`.**Información:** surge cuando los datos se organizan, procesan y analizan para obtener resultados que permitan comprender una situación. Por ejemplo, al calcular la proporción de reservas canceladas se obtiene que aproximadamente el 37 % de las reservas del conjunto analizado fueron canceladas.**Conocimiento:** se obtiene al interpretar la información y relacionarla con el problema que se busca resolver. Por ejemplo, si mediante análisis posteriores se descubre que las reservas realizadas con mayor anticipación presentan una proporción de cancelación diferente, este resultado podría aportar conocimiento útil para comprender el comportamiento de los clientes y orientar decisiones comerciales.En este trabajo, el objetivo es transformar los datos disponibles en información relevante y, posteriormente, obtener conocimiento que permita comprender mejor los factores asociados a las cancelaciones hoteleras.

##Ciclo de vida del análisis de datosEl análisis de datos requiere un proceso organizado que permita pasar desde la identificación de un problema hasta la obtención de resultados que puedan utilizarse para tomar decisiones.Para este trabajo se consideran las siguientes etapas:**1. Identificación del problema:** consiste en comprender la situación que se busca analizar y establecer los objetivos del estudio. En este caso, se busca comprender y predecir las cancelaciones de reservas hoteleras.**2. Obtención y comprensión de los datos:** consiste en identificar el origen de los datos, conocer su estructura y comprender qué representa cada variable. En esta primera entrega se realiza una exploración inicial del dataset asignado.**3. Preparación de los datos:** comprende la revisión de valores faltantes, registros duplicados, inconsistencias y posibles valores atípicos. También incluye las transformaciones necesarias para que los datos puedan utilizarse correctamente.**4. Análisis exploratorio:** consiste en aplicar técnicas estadísticas y herramientas de visualización para identificar distribuciones, comportamientos, patrones y posibles relaciones entre las variables.**5. Modelado:** en esta etapa se pueden aplicar técnicas estadísticas y algoritmos de aprendizaje automático para construir modelos que permitan explicar o predecir determinados comportamientos. Para este trabajo, el interés principal será evaluar la posibilidad de predecir cancelaciones.**6. Evaluación e interpretación:** consiste en comprobar la calidad de los resultados obtenidos, evaluar el desempeño de los modelos y determinar si las conclusiones responden a las preguntas iniciales.**7. Comunicación de resultados:** finalmente, se presentan los hallazgos mediante explicaciones, gráficos y conclusiones que permitan comprender la información y utilizarla como apoyo para la toma de decisiones.Estas etapas no necesariamente se desarrollan de manera estrictamente lineal. Durante el análisis pueden surgir problemas que requieran volver a etapas anteriores, corregir los datos o replantear determinadas decisiones.

## Desarrollo de la entrega 1

###Vista inicialvisualización de las primeras filas del dataset.

In [ ]:
head(datos)

###Estructura generalcantidad de filas, cantidad de columnas y nombres de las variables.

In [ ]:
dimensiones <- dim(datos)cat("Cantidad de filas:", dimensiones[1], "\n")cat("Cantidad de columnas:", dimensiones[2], "\n")cat("Columnas del dataset:\n")print(names(datos))

El dataset presenta las siguientes características, según la ejecución original:- **Cantidad de registros:** 25.000.- **Cantidad de variables:** 32.Las variables identificadas son:`booking_id`, `hotel`, `is_canceled`, `lead_time`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`, `arrival_date`, `stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `babies`, `meal`, `country`, `market_segment`, `distribution_channel`, `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `reserved_room_type`, `assigned_room_type`, `booking_changes`, `deposit_type`, `agent`, `company`, `days_in_waiting_list`, `customer_type`, `adr`, `required_car_parking_spaces` y `total_of_special_requests`.

### Tipos de datos y clasificación conceptual de las variablesEs importante diferenciar dos conceptos:- **Clase de dato en R:** indica cómo se encuentra almacenado un valor dentro del entorno de programación. Algunos ejemplos son `integer`, `numeric` y `character`.- **Tipo conceptual de variable:** representa la naturaleza de la información que contiene y permite clasificarla como numérica, categórica o temporal.Por lo tanto, una variable almacenada como `integer` no necesariamente representa una cantidad numérica. Por ejemplo, `is_canceled` utiliza los valores 0 y 1, pero conceptualmente representa una categoría binaria.

In [ ]:
cat("\nClases de datos en R:\n")print(sapply(datos, class))

A partir de las variables identificadas, se realiza la siguiente clasificación inicial:| Variable | Clasificación conceptual ||---|---|| booking_id | Categórica (identificador) || hotel | Categórica nominal || is_canceled | Categórica binaria || lead_time | Numérica discreta || arrival_date_year | Temporal || arrival_date_month | Temporal || arrival_date_week_number | Temporal || arrival_date_day_of_month | Temporal || arrival_date | Temporal || stays_in_weekend_nights | Numérica discreta || stays_in_week_nights | Numérica discreta || adults | Numérica discreta || children | Numérica discreta || babies | Numérica discreta || meal | Categórica nominal || country | Categórica nominal || market_segment | Categórica nominal || distribution_channel | Categórica nominal || is_repeated_guest | Categórica binaria || previous_cancellations | Numérica discreta || previous_bookings_not_canceled | Numérica discreta || reserved_room_type | Categórica nominal || assigned_room_type | Categórica nominal || booking_changes | Numérica discreta || deposit_type | Categórica nominal || agent | Categórica (identificador) || company | Categórica (identificador) || days_in_waiting_list | Numérica discreta || customer_type | Categórica nominal || adr | Numérica continua || required_car_parking_spaces | Numérica discreta || total_of_special_requests | Numérica discreta |Esta clasificación es inicial y podrá revisarse durante las siguientes etapas, especialmente cuando se realicen transformaciones o se analice con mayor profundidad el significado de cada variable.En particular, las variables `agent` y `company` pueden encontrarse almacenadas como numéricas, pero sus valores representan identificadores y no cantidades sobre las que tenga sentido realizar operaciones aritméticas.Por otro lado, `arrival_date` aparece almacenada como `character` en R, pero conceptualmente representa una fecha. Esto significa que, para realizar operaciones temporales correctamente, podría ser necesario convertirla al formato de fecha correspondiente.

###Caracterización descriptiva inicialSe realiza una primera exploración estadística mediante la función `summary(datos)`, que permite obtener resúmenes generales de las variables y observar sus valores mínimos, máximos, medidas centrales y presencia de datos faltantes, según corresponda.

In [ ]:
summary(datos)

A partir de los resultados obtenidos se destacan las siguientes observaciones:**Variable `is_canceled`**La media de esta variable es de aproximadamente 0,3704. Debido a que utiliza los valores 0 y 1, este resultado representa una proporción de aproximadamente 37,04 % de reservas canceladas, mientras que el 62,96 % restante no presenta cancelaciones.Esta variable resulta especialmente relevante porque constituye la variable objetivo del estudio.**Variable `lead_time`**Presenta una media aproximada de 104 días y una mediana de 69 días, con valores que van desde 0 hasta 629 días.La diferencia entre la media y la mediana sugiere una distribución posiblemente asimétrica, en la que algunas reservas realizadas con mucha anticipación elevan el promedio.**Duración de las estadías**La variable `stays_in_week_nights` presenta una media aproximada de 2,5 noches y una mediana de 2 noches.Por su parte, `stays_in_weekend_nights` presenta una media cercana a 0,9 noches.Estos resultados permiten observar que, en general, las reservas corresponden a estadías relativamente cortas, aunque existen registros con duraciones considerablemente mayores.**Cantidad de huéspedes**La variable `adults` presenta una mediana de 2 adultos por reserva.En cambio, la media de `children` es de aproximadamente 0,10 y la de `babies` es cercana a 0,007.Esto indica que la mayoría de las reservas corresponden principalmente a adultos, mientras que la presencia de niños y bebés es considerablemente menor.**Valores faltantes**Se observa una presencia importante de datos faltantes en algunas variables. Entre ellas, `company` presenta una cantidad elevada de valores ausentes y `agent` también posee una cantidad significativa.En cambio, variables como `hotel`, `meal`, `market_segment` y `deposit_type` no presentan valores ausentes según el resumen obtenido.La presencia de estos valores deberá ser considerada antes de realizar análisis más profundos, ya que podría afectar los resultados y requerir un tratamiento específico.**Variable `adr`**La variable `adr`, correspondiente a la tarifa diaria promedio, presenta una mediana de 94,50 y una media aproximada de 102,06.También se observa un valor mínimo de -6,38, que resulta llamativo para una tarifa hotelera. Este registro deberá ser revisado posteriormente para determinar si corresponde a un error, una situación particular o un problema de registro.**Posibles valores extremos**Algunas variables presentan valores máximos considerablemente superiores a sus valores centrales, como `stays_in_week_nights`, `stays_in_weekend_nights`, `days_in_waiting_list` y `previous_bookings_not_canceled`.Estas diferencias podrían indicar la presencia de valores extremos que deberán analizarse antes de extraer conclusiones definitivas.

###Primeras observaciones y conclusionesA partir de la exploración descriptiva inicial se observa que el conjunto de datos contiene 25.000 reservas hoteleras y 32 variables que incluyen información sobre las características de las reservas, las fechas de llegada, la duración de las estadías, los huéspedes y las cancelaciones.Uno de los primeros resultados relevantes es que aproximadamente el 37 % de las reservas fueron canceladas. Este porcentaje permite dimensionar inicialmente el problema y justifica la importancia de estudiar qué características podrían estar relacionadas con las cancelaciones.También se identifican diferencias entre las medias y medianas de algunas variables, como `lead_time` y `adr`, que sugieren posibles distribuciones asimétricas y justifican la necesidad de profundizar el análisis.Por otro lado, la presencia de valores faltantes y posibles valores atípicos indica que será necesario realizar una etapa de preparación y limpieza antes de avanzar hacia análisis más complejos o modelos predictivos.En esta primera etapa no se establecen relaciones causales ni se afirma que alguna variable sea responsable de las cancelaciones. Los resultados obtenidos representan únicamente una caracterización inicial del conjunto de datos.

###Consideraciones éticas inicialesEn esta primera etapa, los resultados deben interpretarse únicamente como una descripción general del conjunto de datos y no como evidencia de relaciones causales. La existencia de una determinada característica en las reservas no implica necesariamente que dicha característica sea la causa de una cancelación.También es importante considerar la presencia de datos faltantes y posibles valores atípicos, ya que pueden afectar los resultados de los análisis posteriores. Cualquier tratamiento de estos datos deberá estar justificado y evitar modificaciones que puedan introducir sesgos.El análisis debe realizarse respetando la privacidad de las personas involucradas. Aunque el conjunto de datos utilizado no presenta información personal directamente identificable entre las variables observadas, los datos de reservas deben utilizarse únicamente con fines analíticos legítimos y evitando intentar identificar o inferir información personal sobre individuos.Además, si en etapas posteriores se construye un modelo predictivo, será importante evaluar sus posibles sesgos y limitaciones. Una predicción de cancelación no debería interpretarse como una certeza ni utilizarse automáticamente para tomar decisiones que perjudiquen a un cliente.Por último, las conclusiones obtenidas deberán limitarse al conjunto de datos analizado y al contexto en el que fueron recopilados. No sería adecuado generalizar automáticamente estos resultados a todos los hoteles, huéspedes o reservas sin contar con evidencia adicional que permita sostener dicha generalización.

##Entrega 2Esta segunda etapa continúa el trabajo iniciado en la Entrega 1. El objetivo es diagnosticar la calidad del conjunto de datos, justificar las decisiones de limpieza y preparar variables que puedan utilizarse en el análisis posterior de las cancelaciones hoteleras.La limpieza no consiste en eliminar automáticamente todo valor inusual. Cada decisión debe considerar el significado de la variable y el problema que se desea estudiar. Por eso, se conservarán variables relevantes para explicar las cancelaciones y se registrará cada modificación en una bitácora.**Criterios incorporados a partir de la devolución docente:**- Se conservan `lead_time` y `stays_in_week_nights`.- Se incorporan variables derivadas.- Se documentan las transformaciones en una bitácora.- Se revisa `adr` distinguiendo valores negativos, ceros y valores extremos, y se informa la cantidad exacta de filas afectadas mediante el propio código.

###Carga del dataset original

In [ ]:
ruta_dataset <- ruta_csv_compartidadf_original <- read.csv(  ruta_dataset,  stringsAsFactors = FALSE,  na.strings = c("", "NA"))df_hoteles <- df_originalcat("Dimensiones originales:", nrow(df_hoteles), "filas y",    ncol(df_hoteles), "columnas\\n")stopifnot("is_canceled" %in% names(df_hoteles))

###Diagnóstico inicialAntes de modificar los datos, se conserva una copia del dataset original y se revisan dimensiones, estructura, duplicados y valores faltantes. Los resultados se calculan al ejecutar el notebook, evitando consignar cantidades que no hayan sido verificadas.

In [ ]:
dim(df_hoteles)str(df_hoteles)summary(df_hoteles)cat("Filas duplicadas completas:", sum(duplicated(df_hoteles)), "\\n")contar_faltantes <- function(x) {  if (is.character(x)) {    sum(is.na(x) | trimws(x) == "")  } else {    sum(is.na(x))  }}faltantes_iniciales <- sapply(df_hoteles, contar_faltantes)sort(faltantes_iniciales[faltantes_iniciales > 0], decreasing = TRUE)

###Bitácora de limpieza y transformaciónLa bitácora registra el problema detectado, la variable afectada, la decisión, su justificación y la cantidad de registros antes y después de la operación. Así se puede auditar el proceso y comprobar que las explicaciones coincidan con los resultados.

In [ ]:
bitacora <- data.frame(  etapa = character(),  problema = character(),  variable = character(),  decision = character(),  justificacion = character(),  filas_antes = integer(),  filas_despues = integer(),  filas_afectadas = integer(),  stringsAsFactors = FALSE)registrar <- function(etapa, problema, variable, decision,                      justificacion, antes, despues) {  bitacora[nrow(bitacora) + 1, ] <<- list(    etapa, problema, variable, decision, justificacion,    as.integer(antes), as.integer(despues),    as.integer(antes - despues)  )}

### Registros duplicadosSe identifican duplicados completos. Solo se eliminan las filas que sean idénticas en todas sus columnas. No se eliminan reservas que compartan atributos, porque pueden ser observaciones distintas.

In [ ]:
filas_antes <- nrow(df_hoteles)duplicados <- sum(duplicated(df_hoteles))if (duplicados > 0) {  df_hoteles <- df_hoteles[!duplicated(df_hoteles), ]}registrar(  "Limpieza", "Registros completamente duplicados", "Todas las variables",  "Eliminar duplicados exactos",  "Una fila idéntica en todas sus columnas constituye una repetición del mismo registro.",  filas_antes, nrow(df_hoteles))cat("Duplicados detectados y eliminados:", duplicados, "\\n")

### Selección de variablesSe elimina `booking_id` porque es un identificador único y no describe una característica de la reserva. Se eliminan las columnas de componentes de fecha (`arrival_date_year`, `arrival_date_month`, `arrival_date_week_number` y `arrival_date_day_of_month`) porque se conservará `arrival_date` y se derivarán de ella el año y el mes.Se conservan `lead_time` y `stays_in_week_nights`: el primero mide la anticipación de la reserva y puede relacionarse con la cancelación; el segundo permite conocer la duración de la estadía y construir `total_nights`. También se conservan `meal` y `distribution_channel`, ya que son características potencialmente relevantes para estudiar el comportamiento de las reservas.

In [ ]:
columnas_eliminar <- intersect(  c("booking_id", "arrival_date_year", "arrival_date_month",    "arrival_date_week_number", "arrival_date_day_of_month"),  names(df_hoteles))filas_antes <- nrow(df_hoteles)columnas_antes <- ncol(df_hoteles)df_hoteles <- df_hoteles[, !(names(df_hoteles) %in% columnas_eliminar), drop = FALSE]registrar(  "Selección", "Identificador y componentes redundantes de fecha",  paste(columnas_eliminar, collapse = ", "),  "Eliminar columnas indicadas",  "Se conserva arrival_date como fecha completa; lead_time, stays_in_week_nights, meal y distribution_channel permanecen por su posible valor explicativo.",  filas_antes, nrow(df_hoteles))cat("Columnas eliminadas:", paste(columnas_eliminar, collapse = ", "), "\\n")cat("Columnas antes:", columnas_antes,    "| columnas después:", ncol(df_hoteles), "\\n")

### Valores faltantesSe revisan los faltantes por variable y se toman decisiones considerando su significado:- `company`: se elimina porque aproximadamente el 94,2 % de sus registros no tiene dato. La variable quedaría sustentada por una proporción muy pequeña de observaciones.- `agent`: se conserva. Aunque presenta faltantes, el código de agente puede representar un canal de intermediación útil. Se transforma a texto y los faltantes se identifican como `Sin agente`, sin tratarlos como una cantidad numérica.- `country`: los faltantes se reemplazan por `Sin especificar`, conservando las filas y haciendo explícita la ausencia de información.- `children`: los dos faltantes detectados en el diagnóstico inicial se reemplazan por cero, decisión que debe interpretarse como ausencia de niños informados. Se registra expresamente para que pueda revisarse si el contexto del dato lo requiere.La bitácora y los conteos permiten comprobar cuántas celdas se modificaron y si se eliminaron filas o columnas.

In [ ]:
# Eliminar company por su proporción extremadamente alta de faltantesif ("company" %in% names(df_hoteles)) {  n_company_na <- contar_faltantes(df_hoteles$company)  prop_company_na <- n_company_na / nrow(df_hoteles)  filas_antes <- nrow(df_hoteles)  df_hoteles$company <- NULL  registrar(    "Faltantes", "Proporción muy elevada de datos faltantes",    "company", "Eliminar variable",    paste0("Faltantes: ", n_company_na, " (",           round(prop_company_na * 100, 2),           "%). La cobertura resulta insuficiente para un análisis confiable."),    filas_antes, nrow(df_hoteles)  )}# Conservar agent como identificador categórico y explicitar los faltantesif ("agent" %in% names(df_hoteles)) {  n_agent_na <- contar_faltantes(df_hoteles$agent)  df_hoteles$agent <- as.character(df_hoteles$agent)  df_hoteles$agent[is.na(df_hoteles$agent) |                     trimws(df_hoteles$agent) == ""] <- "Sin agente"  registrar(    "Faltantes", "Códigos de agente ausentes",    "agent", "Convertir a categórica y asignar categoría Sin agente",    "El código identifica agentes y no es una magnitud numérica; se conserva la variable y se hace explícita la ausencia.",    nrow(df_hoteles), nrow(df_hoteles)  )}if ("country" %in% names(df_hoteles)) {  n_country_na <- contar_faltantes(df_hoteles$country)  df_hoteles$country[is.na(df_hoteles$country) |                       trimws(df_hoteles$country) == ""] <- "Sin especificar"  registrar(    "Faltantes", "País de origen ausente",    "country", "Reemplazar faltantes por Sin especificar",    "Se preservan las reservas y se distingue la ausencia de información de una categoría real.",    nrow(df_hoteles), nrow(df_hoteles)  )}if ("children" %in% names(df_hoteles)) {  n_children_na <- sum(is.na(df_hoteles$children))  df_hoteles$children[is.na(df_hoteles$children)] <- 0  registrar(    "Faltantes", "Cantidad de niños ausente",    "children", "Reemplazar NA por 0",    "En el diagnóstico inicial se observaron dos faltantes. Se adopta cero como ausencia de niños informados y se deja documentada la imputación.",    nrow(df_hoteles), nrow(df_hoteles)  )}cat("Faltantes restantes por variable:\\n")faltantes_finales_limpieza <- sapply(df_hoteles, contar_faltantes)print(faltantes_finales_limpieza[faltantes_finales_limpieza > 0])

### Revisión de valores de `adr``adr` representa la tarifa diaria promedio. Se diferencian tres situaciones:- **Valores negativos:** no son coherentes con una tarifa diaria promedio y se consideran inválidos.- **Valores iguales a cero:** no se eliminan automáticamente. Pueden representar una tarifa sin cargo, una condición particular o un registro que requiere análisis contextual.- **Valores altos:** se investigan como posibles valores extremos, pero no se eliminan solo por superar un umbral. Un valor atípico no necesariamente es un error.Para corregir la inconsistencia señalada por el docente, el código calcula y muestra la cantidad exacta de registros con `adr` negativo y la cantidad de filas eliminadas. La explicación no fija una cifra manual: el resultado ejecutado es la referencia.

In [ ]:
if ("adr" %in% names(df_hoteles)) {  cat("Resumen de adr antes del tratamiento:\\n")  print(summary(df_hoteles$adr))  cantidad_negativos <- sum(df_hoteles$adr < 0, na.rm = TRUE)  cantidad_ceros <- sum(df_hoteles$adr == 0, na.rm = TRUE)  cantidad_na_adr <- sum(is.na(df_hoteles$adr))  filas_antes <- nrow(df_hoteles)  df_hoteles <- df_hoteles[    is.na(df_hoteles$adr) | df_hoteles$adr >= 0,    , drop = FALSE  ]  filas_despues <- nrow(df_hoteles)  registrar(    "Valores inválidos", "Tarifas diarias promedio negativas",    "adr", "Eliminar únicamente filas con adr < 0",    "Una tarifa promedio negativa no tiene interpretación válida en este contexto. Los ceros se conservan porque pueden responder a situaciones reales que requieren análisis.",    filas_antes, filas_despues  )  cat("Registros con adr negativo detectados:", cantidad_negativos, "\\n")  cat("Registros con adr igual a cero conservados:", cantidad_ceros, "\\n")  cat("Valores faltantes de adr:", cantidad_na_adr, "\\n")  cat("Filas eliminadas por adr negativo:", filas_antes - filas_despues, "\\n")  cat("Filas restantes:", filas_despues, "\\n")} else {  warning("No se encontró la variable adr.")}

### Reservas sin huéspedes informadosSe identifica la cantidad de reservas en las que `adults`, `children` y `babies` son simultáneamente cero. No se eliminan automáticamente: una reserva cancelada o modificada podría presentar esta combinación y eliminarla podría sesgar el análisis de `is_canceled`.En su lugar, se crea una variable indicadora para poder estudiar estos casos y decidir más adelante, con evidencia, si corresponde excluirlos.

In [ ]:
if (all(c("adults", "children", "babies") %in% names(df_hoteles))) {  df_hoteles$reserva_sin_huespedes <-    df_hoteles$adults == 0 &    df_hoteles$children == 0 &    df_hoteles$babies == 0  cat("Reservas sin huéspedes informados:",      sum(df_hoteles$reserva_sin_huespedes, na.rm = TRUE), "\\n")}

### Detección de valores atípicosSe utiliza el criterio del rango intercuartílico (IQR): un valor se marca como atípico si está por debajo de Q1 − 1,5 × IQR o por encima de Q3 + 1,5 × IQR.La detección no implica eliminación. Variables como `lead_time`, duración de estadía, cancelaciones previas y `adr` pueden presentar distribuciones asimétricas y valores altos reales. Se informa la cantidad de observaciones señaladas para su revisión.

In [ ]:
detectar_outliers <- function(x) {  x <- x[is.finite(x)]  if (length(x) < 4) return(integer(0))  q1 <- quantile(x, 0.25, na.rm = TRUE, names = FALSE)  q3 <- quantile(x, 0.75, na.rm = TRUE, names = FALSE)  iqr <- q3 - q1  which(x < (q1 - 1.5 * iqr) | x > (q3 + 1.5 * iqr))}variables_numericas_revision <- intersect(  c("lead_time", "stays_in_weekend_nights", "stays_in_week_nights",    "previous_cancellations", "previous_bookings_not_canceled",    "booking_changes", "days_in_waiting_list", "adr"),  names(df_hoteles))resumen_outliers <- lapply(variables_numericas_revision, function(v) {  x <- df_hoteles[[v]]  x_valido <- x[!is.na(x)]  q1 <- quantile(x_valido, 0.25, na.rm = TRUE)  q3 <- quantile(x_valido, 0.75, na.rm = TRUE)  iqr <- q3 - q1  data.frame(    variable = v,    cantidad_atipicos = sum(x_valido < (q1 - 1.5 * iqr) |                            x_valido > (q3 + 1.5 * iqr)),    limite_inferior = as.numeric(q1 - 1.5 * iqr),    limite_superior = as.numeric(q3 + 1.5 * iqr)  )})resumen_outliers <- bind_rows(resumen_outliers)print(resumen_outliers)cat("Los valores atípicos se conservan en esta etapa; no se eliminan sin una justificación contextual.\\n")

###Transformación y creación de variables derivadasLas variables derivadas permiten representar características que no estaban expresadas directamente en una única columna. Se conservan las variables originales cuando siguen siendo útiles para interpretar los resultados.

###Fecha de llegada: conversión y componentesSe convierte `arrival_date` al tipo fecha y se derivan el año y el mes. Esto permite realizar agrupaciones temporales sin depender de columnas redundantes del archivo original.

In [ ]:
if ("arrival_date" %in% names(df_hoteles)) {  fecha_convertida <- as.Date(df_hoteles$arrival_date, format = "%Y-%m-%d")  cat("Fechas que no pudieron convertirse:",      sum(is.na(fecha_convertida) & !is.na(df_hoteles$arrival_date)), "\\n")  df_hoteles$arrival_date <- fecha_convertida  df_hoteles$arrival_year <- year(df_hoteles$arrival_date)  df_hoteles$arrival_month <- month(df_hoteles$arrival_date,                                    label = TRUE, abbr = FALSE)}

### Duración total de la estadíaSe crea `total_nights`, sumando las noches de semana y de fin de semana. Esta variable facilita estudiar la duración completa de cada reserva. Se mantienen las dos variables originales para permitir análisis diferenciados.

In [ ]:
if (all(c("stays_in_week_nights", "stays_in_weekend_nights") %in%          names(df_hoteles))) {  df_hoteles$total_nights <-    df_hoteles$stays_in_week_nights +    df_hoteles$stays_in_weekend_nights}

### Cantidad total de huéspedesSe crea `total_guests`, sumando adultos, niños y bebés. La variable representa el tamaño total del grupo asociado a la reserva.

In [ ]:
if (all(c("adults", "children", "babies") %in% names(df_hoteles))) {  df_hoteles$total_guests <-    df_hoteles$adults + df_hoteles$children + df_hoteles$babies}

###Indicador de reserva anticipadaSe crea `early_booking`, una variable binaria que indica si la reserva se realizó con más de 90 días de anticipación. El umbral de 90 días se adopta como una segmentación operativa para explorar la relación entre anticipación y cancelación; no implica que ese límite sea una regla universal.

In [ ]:
if ("lead_time" %in% names(df_hoteles)) {  df_hoteles$early_booking <- ifelse(    is.na(df_hoteles$lead_time), NA,    ifelse(df_hoteles$lead_time > 90, 1L, 0L)  )}

###Estandarización de `adr`Se crea `adr_standard`, que expresa la tarifa en unidades de desvío estándar respecto de la media. Se conserva `adr` original para mantener la interpretación en su unidad de medida.La estandarización no elimina valores atípicos ni corrige errores: solo cambia la escala.

if ("adr" %in% names(df_hoteles)) {  media_adr <- mean(df_hoteles$adr, na.rm = TRUE)  desvio_adr <- sd(df_hoteles$adr, na.rm = TRUE)  if (is.finite(desvio_adr) && desvio_adr > 0) {    df_hoteles$adr_standard <- (df_hoteles$adr - media_adr) / desvio_adr  } else {    warning("No se pudo estandarizar adr porque el desvío es cero o no está definido.")  }}

### Codificación binaria de la variable `hotel`Se generan variables indicadoras (0/1) para las categorías de hotel. Se conserva la variable categórica original, porque es más fácil de interpretar y puede ser útil en tablas y gráficos. Las columnas binarias pueden facilitar el uso posterior de determinados modelos.

In [ ]:
if ("hotel" %in% names(df_hoteles)) {  niveles_hotel <- sort(unique(na.omit(df_hoteles$hotel)))  for (nivel in niveles_hotel) {    nombre_columna <- paste0("hotel_", make.names(nivel))    df_hoteles[[nombre_columna]] <- as.integer(df_hoteles$hotel == nivel)  }}

### Registro de las transformacionesSe incorporan a la bitácora las variables derivadas y las transformaciones realizadas. En esta sección también se verifica que las variables originales relevantes continúen presentes.

In [ ]:
registrar(  "Transformación", "Representación temporal",  "arrival_date, arrival_year, arrival_month",  "Convertir fecha y derivar año y mes",  "Permite analizar la distribución temporal de las llegadas y conserva la fecha completa.",  nrow(df_hoteles), nrow(df_hoteles))registrar(  "Transformación", "Duración total de la reserva",  "total_nights",  "Sumar noches de semana y fin de semana",  "Resume la duración total de la estadía; se conservan las variables que componen la suma.",  nrow(df_hoteles), nrow(df_hoteles))registrar(  "Transformación", "Tamaño del grupo",  "total_guests",  "Sumar adultos, niños y bebés",  "Permite estudiar el tamaño total del grupo de huéspedes.",  nrow(df_hoteles), nrow(df_hoteles))registrar(  "Transformación", "Anticipación de la reserva",  "early_booking",  "Crear indicador: lead_time > 90",  "Segmenta las reservas según un umbral operativo de anticipación.",  nrow(df_hoteles), nrow(df_hoteles))registrar(  "Transformación", "Escala de tarifa",  "adr_standard",  "Estandarizar adr mediante media y desvío estándar",  "Permite comparar la tarifa en una escala común; se conserva adr original.",  nrow(df_hoteles), nrow(df_hoteles))registrar(  "Transformación", "Codificación de tipo de hotel",  "hotel_*",  "Crear indicadores binarios por categoría",  "Facilita la utilización de la variable categórica en modelos; se conserva hotel original.",  nrow(df_hoteles), nrow(df_hoteles))print(bitacora)

###Verificación del conjunto preparadoSe revisan las dimensiones finales, las variables disponibles, los faltantes y los resúmenes estadísticos. Esta comprobación permite detectar errores introducidos durante la limpieza o transformación.

In [ ]:
cat("Dimensiones finales:", nrow(df_hoteles), "filas y",    ncol(df_hoteles), "columnas\\n")cat("\\nVariables finales:\\n")print(names(df_hoteles))cat("\\nFaltantes finales por variable:\\n")faltantes_finales <- sapply(df_hoteles, contar_faltantes)print(faltantes_finales[faltantes_finales > 0])cat("\\nResumen de variables principales:\\n")variables_resumen <- intersect(  c("is_canceled", "lead_time", "stays_in_week_nights",    "stays_in_weekend_nights", "total_nights", "total_guests",    "adr", "adr_standard"),  names(df_hoteles))print(summary(df_hoteles[, variables_resumen, drop = FALSE]))cat("\\nDistribución de la variable objetivo is_canceled:\\n")print(table(df_hoteles$is_canceled, useNA = "ifany"))print(prop.table(table(df_hoteles$is_canceled, useNA = "no")))

###Controles de conservación de variables relevantesSe comprueba explícitamente que las variables señaladas por el docente no hayan sido eliminadas.

In [ ]:
variables_que_deben_conservarse <- c(  "lead_time", "stays_in_week_nights", "is_canceled")variables_ausentes <- setdiff(variables_que_deben_conservarse,                              names(df_hoteles))if (length(variables_ausentes) == 0) {  cat("Control correcto: se conservaron lead_time, stays_in_week_nights e is_canceled.\\n")} else {  stop(paste("Faltan variables requeridas:",             paste(variables_ausentes, collapse = ", ")))}variables_derivadas_esperadas <- c(  "total_nights", "total_guests", "early_booking",  "adr_standard", "arrival_year", "arrival_month")cat("Variables derivadas presentes:\\n")print(intersect(variables_derivadas_esperadas, names(df_hoteles)))

###Bitácora final y resumen de modificacionesLa bitácora completa permite consultar cada decisión y sus motivos. Las filas afectadas se calculan a partir de las dimensiones antes y después de cada operación; en las transformaciones que no eliminan filas, el valor es cero. Para los reemplazos de valores faltantes se informa la cantidad detectada en las salidas específicas de cada sección.

In [ ]:
print(bitacora, row.names = FALSE)cat("\\nResumen de filas por etapa:\\n")print(bitacora %>%        group_by(etapa) %>%        summarise(          filas_afectadas = sum(filas_afectadas),          operaciones_registradas = n(),          .groups = "drop"        ))cat("\\nFilas originales:", nrow(df_original), "\\n")cat("Filas finales:", nrow(df_hoteles), "\\n")cat("Diferencia neta de filas:", nrow(df_original) - nrow(df_hoteles), "\\n")

###ConclusionesEn esta entrega se realizó un proceso de diagnóstico, limpieza y transformación del dataset de reservas hoteleras. Las decisiones se tomaron considerando el significado de las variables y su posible relación con la cancelación de reservas.A diferencia de una limpieza basada únicamente en eliminar columnas o valores inusuales, se conservaron variables relevantes como `lead_time` y `stays_in_week_nights`. También se mantuvieron `meal`, `distribution_channel` y `agent`, ya que pueden aportar información para caracterizar las reservas. Se eliminó `booking_id` por su función identificadora y se reemplazaron las columnas de componentes de fecha por una fecha completa, de la que luego se derivaron año y mes.En el tratamiento de faltantes se eliminó `company` debido a su proporción muy elevada de valores ausentes. Se conservaron otras variables con faltantes mediante categorías explícitas o imputación documentada. En `adr` se eliminaron exclusivamente los valores negativos, se conservaron los ceros y se revisaron los valores extremos sin eliminarlos automáticamente. La cantidad exacta de registros eliminados se obtiene de la ejecución del código y queda reflejada en la bitácora.Finalmente, se crearon variables derivadas como `total_nights`, `total_guests`, `early_booking`, `adr_standard`, `arrival_year` y `arrival_month`, además de indicadores binarios para el tipo de hotel. Estas transformaciones amplían las posibilidades de análisis sin perder las variables originales necesarias para interpretar los resultados.El conjunto resultante queda preparado para continuar con el análisis exploratorio y estudiar la relación entre las características de las reservas y la variable objetivo `is_canceled`.**Importante:** los resultados numéricos definitivos deben tomarse de la ejecución completa de este notebook sobre el CSV original. No se fijan manualmente cantidades de filas eliminadas ni estadísticas que puedan variar según el archivo cargado.

---# ENTREGA 3 — Análisis inicial y visualizacionesContenido incorporado desde el notebook original **Aguillon_Fernando_Comision5_Entrega3 (1).ipynb**.

# Trabajo Práctico Integrador - Introducción al Análisis de Datos**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.> Esta plantilla está pensada para construir notebooks independientes y reproducibles. Cada entrega debe poder ejecutarse desde cero, cargando nuevamente el dataset original asignado a la comisión.

## 0.Datos de identificación- **Apellido y nombre:** Aguillón, Fernando- **Comisión:** 5- **Dataset asignado:** E- **Entrega:** 2- **Fecha:** 06/09/2026

##1.Importación de libreríasImportar solamente las librerías necesarias para la entrega que se está desarrollando.

In [ ]:
# Librerías básicasinstall.packages("GGally")library(ggplot2)library(GGally)

##2.Carga del datasetCargar el archivo CSV correspondiente a la comisión. El notebook debe funcionar desde el dataset original, no desde objetos creados en otros notebooks.

In [ ]:
df_hoteles <- read.csv(ruta_csv_compartida)

##3. Limpieza realizada en la entrega 2 con las correcciones correspondientes

In [ ]:
df_hoteles <- df_hoteles[, !(names(df_hoteles) %in% c('booking_id', 'meal', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'arrival_date_week_number' , 'distribution_channel'))]df_hoteles <- df_hoteles[, !(names(df_hoteles) %in% c('company', 'agent'))]df_hoteles$country[is.na(df_hoteles$country) | df_hoteles$country == ""] <- "S/C"df_hoteles$children[is.na(df_hoteles$children)] <- 0df_hoteles <- df_hoteles[!(df_hoteles$adr < 0 | df_hoteles$adr == 0), ]df_hoteles <- df_hoteles[!(df_hoteles$adults == 0 & df_hoteles$children == 0 & df_hoteles$babies == 0), ]

##4. Visualizaciones

###4.1 Cancelaciones según tipo de hotel¿Que reacion hay entre las reservas canceladas y el tipo de hotel elegido?Se utiliza un grafico de conteo porque permite comparar la cantidad de reservas canceladas y no canceladas segun el tipo de hotel . Es adecuado porque ambas variables son categoricas.

In [ ]:
ggplot(df_hoteles, aes(  x = hotel,  fill = factor(is_canceled))) +  geom_bar() +  labs(    title = "Cantidad de reservas canceladas según tipo de hotel",    x = "Tipo de hotel",    y = "Cantidad de reservas",    fill = "Estado de la reserva"  ) +  scale_fill_discrete(    labels = c(      "0" = "No cancelada",      "1" = "Cancelada"    )  ) +  theme_minimal()

Se observa que el `City hotel` concentra una mayor cantidad de reservas, tanto canceladas como no canceladas, pero de manera ligeramente equilibrada. En cambio se ve como `Resort Hotel` tiene un porcentaje menor a un tercio de reservas canceladas. Esto permite comparar el comportamineto de las cancelaciones entre ambos tipos de hotel, dando como resultado que se debe observan el comportamiento en especial de las reservas en `City hotel`.

##4.2 Variable cuantitativa y cancelación¿Existe una diferencia entre la tarifa promedio por noche entre las reservas canceladas y no canceladas?Se eligio un diagrama de caja (Box Plot) porque permite comparar la distribucion de uan variable numerica continua, en este caso `adr`, entre dos grupos definidos por una variable categorica, `is_canceled`. Ademas permite observar la mediana, la dispersion y posibles valores atipicos.

In [ ]:
library(ggplot2)ggplot(df_hoteles, aes(  x = factor(is_canceled),  y = adr)) +  geom_boxplot() +  labs(    title = "Distribución de la tarifa promedio (ADR) según estado de cancelación",    x = "Estado de la reserva",    y = "Tarifa promedio por noche (ADR)"  ) +  scale_x_discrete(    labels = c(      "0" = "No cancelada",      "1" = "Cancelada"    )  ) +  theme_minimal()

El `adr` presenta una distribucion similar entre reservas canceladas y no canceladdas, pero con una ligera diferencia entre su mediana apensas perceptible. Tambien se ve que ambas opciones presentan valores atipicos similares. En general se puede decir que la tarifa promedio por noche no indica una clara razon de las cancelaciones.

##4.3 Una pregunta propia de análisis¿Existe una relacion entre la cantidad de dias de anticipacion de una reserva y la probabilidad de ser cancelada?Se selecciono un diagrama de caja porque permite comparar la distribucion de los dias de anticipacion de las reservas entre aquellas que fueron canceladas y las que no.

In [ ]:
ggplot(df_hoteles, aes(  x = factor(is_canceled),  y = lead_time)) +  geom_boxplot() +  labs(    title = "Días de anticipación de la reserva según estado de cancelación",    x = "Estado de la reserva",    y = "Días de anticipación (Lead Time)"  ) +  scale_x_discrete(    labels = c(      "0" = "No cancelada",      "1" = "Cancelada"    )  ) +  theme_minimal()

Las reservas canceladas presentan una media notablemente mayor a las no canceladas y una mayor dispersion. Ademas, se observan numerosos valores atipicos, especialmente el las reservas canceladas, con anticipaciones de mas de 600 dias. Esto sugiere una idea al respecto `A mayor anticipacion de la reserva, existe una mayor tendencia a la cancelacion`.

---# ENTREGA 4 — Análisis exploratorio de datos (EDA)Contenido incorporado desde el notebook original **Aguillon_Fernando_Comision5_Entrega4.ipynb**.

# Trabajo Práctico Integrador – Introducción al Análisis de Datos## Entrega 4: Análisis Exploratorio de Datos**Apellido y nombre:** Aguillón, Fernando  **Comisión:** 5  **Dataset:** E – reservas hoteleras  **Entrega:** 4  **Recorrido elegido:** SQL + R

## 1. Objetivo y continuidadSe estudian diferencias entre reservas canceladas y no canceladas. La variable objetivo es `is_canceled`. La unidad de análisis es **una reserva hotelera por fila**: cada registro representa una reserva, no una persona ni un hotel.Se retoman los criterios corregidos de la Entrega 2: conservar `lead_time`, `stays_in_week_nights`, `meal`, `distribution_channel` y `agent`; reemplazar faltantes de `children` por cero; identificar faltantes de `agent` y `country`; eliminar solo `adr < 0`; conservar `adr = 0`, valores extremos y reservas sin huéspedes, marcando estas últimas sin eliminarlas. Se quitan duplicados exactos, `booking_id` y `company`.### Preguntas formuladas1. ¿La proporción de cancelaciones difiere entre City Hotel y Resort Hotel?2. ¿La distribución de `adr` difiere entre reservas canceladas y no canceladas?3. ¿La distribución de `lead_time` difiere entre reservas canceladas y no canceladas?

## 2. Librerías y entorno SQL

In [ ]:
options(HTTPUserAgent = sprintf(  "R/%s R (%s)",  getRversion(),  paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))install.packages(  "duckdb",  repos = "https://p3m.dev/cran/__linux__/manylinux_2_28/latest/")library(duckdb)packageVersion("duckdb")

In [ ]:
library(DBI)library(duckdb)library(dplyr)library(ggplot2)library(scales)cat("Todos los paquetes se cargaron correctamente.\n")

## 3. Carga del CSV original

In [ ]:
ruta_csv <- ruta_csv_compartidaif (!file.exists(ruta_csv)) {  stop("No se encontró el CSV. Subí 'hotel booking TPI grupo E.csv' a /content/ y volvé a ejecutar.")}datos_originales <- read.csv(  ruta_csv, stringsAsFactors = FALSE,  na.strings = c("", "NA"), check.names = FALSE)cat("Filas originales:", nrow(datos_originales), "\\n")cat("Columnas originales:", ncol(datos_originales), "\\n")

## 4. Preparación mínima y documentadaSe aplican las decisiones corregidas de la Entrega 2.

In [ ]:
df <- datos_originalesfilas_iniciales <- nrow(df)duplicados <- sum(duplicated(df))faltantes_children <- if ("children" %in% names(df)) sum(is.na(df$children)) else NAadr_negativos <- if ("adr" %in% names(df)) sum(df$adr < 0, na.rm = TRUE) else NAadr_ceros <- if ("adr" %in% names(df)) sum(df$adr == 0, na.rm = TRUE) else NAdf <- df[!duplicated(df), ]df <- df[, !(names(df) %in% intersect(c("booking_id", "company"), names(df))), drop = FALSE]if ("children" %in% names(df)) df$children[is.na(df$children)] <- 0if ("agent" %in% names(df)) {  df$agent <- as.character(df$agent)  df$agent[is.na(df$agent) | trimws(df$agent) == ""] <- "Sin agente"}if ("country" %in% names(df)) {  df$country <- as.character(df$country)  df$country[is.na(df$country) | trimws(df$country) == ""] <- "Sin especificar"}if ("adr" %in% names(df)) df <- df[is.na(df$adr) | df$adr >= 0, , drop = FALSE]if (all(c("adults", "children", "babies") %in% names(df))) {  df$reserva_sin_huespedes <- with(df, adults == 0 & children == 0 & babies == 0)}if (all(c("stays_in_week_nights", "stays_in_weekend_nights") %in% names(df))) {  df$total_nights <- df$stays_in_week_nights + df$stays_in_weekend_nights}if (all(c("adults", "children", "babies") %in% names(df))) {  df$total_guests <- df$adults + df$children + df$babies}cat("Duplicados exactos eliminados:", duplicados)cat("Faltantes originales de children:", faltantes_children)cat("ADR negativos detectados:", adr_negativos)cat("ADR igual a cero conservados antes del filtro:", adr_ceros)cat("Filas finales:", nrow(df))cat("Diferencia de filas:", filas_iniciales - nrow(df))

## 5. Revisión inicial### 5.1 Variable objetivo y controles`is_canceled` representa el estado de cancelación: `0` = no cancelada y `1` = cancelada. Se verifica que los valores presentes coincidan con esa codificación.

In [ ]:
requeridas <- c("hotel", "is_canceled", "adr", "lead_time")faltantes <- setdiff(requeridas, names(df))if (length(faltantes) > 0) stop(paste("Faltan columnas:", paste(faltantes, collapse = ", ")))df$is_canceled <- as.integer(as.character(df$is_canceled))df$hotel <- as.character(df$hotel)df$adr <- as.numeric(df$adr)df$lead_time <- as.numeric(df$lead_time)if (!all(na.omit(unique(df$is_canceled)) %in% c(0L, 1L))) {  stop("is_canceled contiene valores distintos de 0 y 1. Revisar codificación.")}

### 5.2 Tabla SQL y estructuraSe crea la tabla `reservas`. Las consultas siguientes verifican columnas, tipos, cantidad de filas y primeras observaciones.

In [ ]:
conexion <- DBI::dbConnect(duckdb::duckdb(), dbdir = ":memory:")DBI::dbWriteTable(conexion, "reservas", df, overwrite = TRUE)estructura <- DBI::dbGetQuery(conexion, "DESCRIBE reservas")estructuraDBI::dbGetQuery(conexion, "SELECT COUNT(*) AS cantidad_filas FROM reservas")DBI::dbGetQuery(conexion, "SELECT * FROM reservas LIMIT 10")

### 5.3 Distribución inicial de `is_canceled`La tabla informa frecuencias absolutas y porcentajes. El porcentaje se calcula sobre los registros agrupados por estado.

In [ ]:
frecuencia_objetivo <- DBI::dbGetQuery(  conexion,  "  SELECT is_canceled,         CASE WHEN is_canceled = 0 THEN 'No cancelada'              WHEN is_canceled = 1 THEN 'Cancelada'              ELSE 'Sin dato' END AS estado,         COUNT(*) AS cantidad,         ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje  FROM reservas  GROUP BY is_canceled  ORDER BY is_canceled  ")frecuencia_objetivo

In [ ]:
ggplot(frecuencia_objetivo, aes(x = estado, y = cantidad, fill = estado)) +  geom_col(show.legend = FALSE) +  labs(title = "Distribución del estado de las reservas",       x = "Estado", y = "Cantidad de reservas") +  theme_minimal()

**Interpretación:** la tabla muestra cuántas reservas corresponden a cada estado y qué porcentaje representan. El gráfico permite observar si las categorías están equilibradas o si una concentra más casos.

### 5.4 Variables seleccionadas y resúmenes individualesSe seleccionan cuatro variables centrales:- `is_canceled`: binaria; variable objetivo.- `hotel`: categórica; permite comparar tipos de hotel.- `adr`: cuantitativa; tarifa promedio por noche.- `lead_time`: cuantitativa; días entre la reserva y la llegada.**Resumen cuantitativo de `lead_time`:**

In [ ]:
DBI::dbGetQuery(  conexion,  "  SELECT COUNT(lead_time) AS cantidad,         AVG(lead_time) AS media,         MEDIAN(lead_time) AS mediana,         STDDEV_SAMP(lead_time) AS desvio,         QUANTILE_CONT(lead_time, 0.25) AS q1,         QUANTILE_CONT(lead_time, 0.75) AS q3,         MIN(lead_time) AS minimo,         MAX(lead_time) AS maximo  FROM reservas  ")

**Frecuencia de la variable categórica `hotel`:**

In [ ]:
frecuencia_hotel <- DBI::dbGetQuery(  conexion,  "  SELECT hotel, COUNT(*) AS cantidad,         ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje  FROM reservas  GROUP BY hotel  ORDER BY cantidad DESC  ")frecuencia_hotel

### 5.5 Visualizaciones individuales inicialesLos histogramas permiten observar la distribución de las variables cuantitativas; el gráfico de barras, la frecuencia de la variable categórica.

In [ ]:
ggplot(df, aes(x = hotel)) +  geom_bar() +  labs(title = "Reservas por tipo de hotel", x = "Hotel", y = "Cantidad") +  theme_minimal()

In [ ]:
ggplot(df, aes(x = lead_time)) +  geom_histogram(bins = 30, na.rm = TRUE) +  labs(title = "Distribución de lead_time",       x = "Días de anticipación", y = "Frecuencia") +  theme_minimal()

In [ ]:
ggplot(df, aes(x = adr)) +  geom_histogram(bins = 30, na.rm = TRUE) +  labs(title = "Distribución de ADR",       x = "Tarifa promedio por noche", y = "Frecuencia") +  theme_minimal()

**Interpretación inicial:** el gráfico de barras compara el volumen de reservas por hotel. Los histogramas muestran la forma de las distribuciones de anticipación y tarifa, permitiendo observar asimetrías y posibles valores alejados del centro. Son distribuciones individuales; todavía no comparan los estados de cancelación.

### 5.6 Preguntas exploratorias formuladas1. ¿La proporción de cancelaciones difiere entre City Hotel y Resort Hotel?2. ¿La distribución de `adr` presenta diferencias entre reservas canceladas y no canceladas?3. ¿La distribución de `lead_time` presenta diferencias entre reservas canceladas y no canceladas?

## 6. Comparaciones mediante SQLCada consulta informa tamaños de grupo y medidas descriptivas. Las proporciones se calculan dentro de cada grupo, no sobre el total general.

### Pregunta 1. Tipo de hotel y cancelación¿La proporción de cancelaciones difiere según el tipo de hotel?

In [ ]:
comparacion_hotel <- DBI::dbGetQuery(  conexion,  "  SELECT hotel,         COUNT(*) AS total_reservas,         SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END) AS canceladas,         SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END) AS no_canceladas,         ROUND(100.0 * SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END)               / COUNT(*), 2) AS porcentaje_canceladas  FROM reservas  WHERE is_canceled IN (0, 1)  GROUP BY hotel  ORDER BY hotel  ")comparacion_hotel

In [ ]:
ggplot(comparacion_hotel,       aes(x = hotel, y = porcentaje_canceladas, fill = hotel)) +  geom_col(show.legend = FALSE) +  labs(title = "Porcentaje de cancelaciones por tipo de hotel",       x = "Tipo de hotel", y = "Canceladas (%)") +  theme_minimal()

**Interpretación:** Tener más reservas no significa necesariamente tener una mayor proporción de cancelaciones. La diferencia observada no demuestra causalidad.

### Pregunta 2. ADR y cancelación¿La distribución de la tarifa promedio por noche difiere entre reservas canceladas y no canceladas?

In [ ]:
comparacion_adr <- DBI::dbGetQuery(  conexion,  "  SELECT is_canceled,         CASE WHEN is_canceled = 1 THEN 'Cancelada' ELSE 'No cancelada' END AS estado,         COUNT(adr) AS cantidad,         AVG(adr) AS media,         MEDIAN(adr) AS mediana,         STDDEV_SAMP(adr) AS desvio,         QUANTILE_CONT(adr, 0.25) AS q1,         QUANTILE_CONT(adr, 0.75) AS q3,         MIN(adr) AS minimo,         MAX(adr) AS maximo  FROM reservas  WHERE is_canceled IN (0, 1) AND adr IS NOT NULL  GROUP BY is_canceled  ORDER BY is_canceled  ")comparacion_adr

In [ ]:
ggplot(df %>% filter(is_canceled %in% c(0, 1)),       aes(x = factor(is_canceled, levels = c(0, 1),                      labels = c("No cancelada", "Cancelada")),           y = adr)) +  geom_boxplot(na.rm = TRUE) +  labs(title = "Distribución de ADR según cancelación",       x = "Estado de la reserva", y = "ADR") +  theme_minimal()

**Interpretación:** Las medianas difieren, cuánto se superponen los rangos intercuartílicos y si hay valores extremos. Una mediana cercana no alcanza para afirmar que ADR no se relaciona con cancelaciones. La comparación es descriptiva.

### Pregunta 3. Lead time y cancelación¿La anticipación con la que se realiza la reserva difiere entre reservas canceladas y no canceladas?

In [ ]:
comparacion_lead <- DBI::dbGetQuery(  conexion,  "  SELECT is_canceled,         CASE WHEN is_canceled = 1 THEN 'Cancelada' ELSE 'No cancelada' END AS estado,         COUNT(lead_time) AS cantidad,         AVG(lead_time) AS media,         MEDIAN(lead_time) AS mediana,         STDDEV_SAMP(lead_time) AS desvio,         QUANTILE_CONT(lead_time, 0.25) AS q1,         QUANTILE_CONT(lead_time, 0.75) AS q3,         MIN(lead_time) AS minimo,         MAX(lead_time) AS maximo  FROM reservas  WHERE is_canceled IN (0, 1) AND lead_time IS NOT NULL  GROUP BY is_canceled  ORDER BY is_canceled  ")comparacion_lead

In [ ]:
ggplot(df %>% filter(is_canceled %in% c(0, 1)),       aes(x = factor(is_canceled, levels = c(0, 1),                      labels = c("No cancelada", "Cancelada")),           y = lead_time)) +  geom_boxplot(na.rm = TRUE) +  labs(title = "Distribución de lead_time según cancelación",       x = "Estado de la reserva", y = "Días de anticipación") +  theme_minimal()

**Interpretación:** En el boxplot la línea central es la **mediana**, la caja representa el rango intercuartílico y los puntos alejados pueden ser valores atípicos. Una mediana mayor en reservas canceladas sugeriría asociación con una mayor anticipación, pero el boxplot no muestra directamente una probabilidad de cancelación ni demuestra causalidad.

## 7. Patrones y casos llamativosSe conservan los casos inusuales. Estas consultas permiten localizarlos para una revisión contextual, sin eliminarlos automáticamente.

In [ ]:
# Reservas con anticipación de 365 días o máscasos_lead <- DBI::dbGetQuery(  conexion,  "  SELECT hotel, lead_time, adr, is_canceled  FROM reservas  WHERE lead_time >= 365  ORDER BY lead_time DESC  LIMIT 20  ")casos_lead

In [ ]:
# ADR igual a cero y valores iguales o superiores al percentil 99casos_adr <- DBI::dbGetQuery(  conexion,  "  SELECT hotel, adr, lead_time, is_canceled  FROM reservas  WHERE adr = 0     OR adr >= (SELECT QUANTILE_CONT(adr, 0.99)                FROM reservas WHERE adr IS NOT NULL)  ORDER BY adr  LIMIT 30  ")casos_adr

**Interpretación:** las consultas identifican reservas con anticipación muy alta o tarifas nulas/extremas. Un valor inusual no es necesariamente un error; debe revisarse junto con el contexto y otras variables. No se elimina ni modifica ningún registro por aparecer aquí.

## 8. Variables candidatas para etapas posteriores| Variable | Justificación ||---|---|| `hotel` | Permite comparar tipos de establecimiento y sus tasas observadas de cancelación. || `lead_time` | Puede presentar distribuciones distintas entre los estados. || `adr` | Permite evaluar diferencias en nivel y variabilidad de tarifas. || `deposit_type` | Puede representar condiciones de garantía de la reserva. || `market_segment` | Caracteriza el segmento de mercado. || `distribution_channel` | Identifica el canal por el que se realizó la reserva. || `previous_cancellations` | Resume antecedentes de cancelaciones. || `customer_type` | Caracteriza el tipo de cliente. || `total_nights` | Resume la duración de la estadía. |Su utilidad deberá evaluarse posteriormente, teniendo en cuenta la calidad de los datos y evitando información que no estaría disponible al momento de predecir.

## 9. Conclusiones exploratorias: tres hallazgosCompletar luego de ejecutar las consultas. Cada hallazgo debe incluir valores numéricos reales obtenidos del SQL.**Hallazgo 1 – Tipo de hotel**   En el conjunto de datos se observa una diferencia en los porcentajes de cancelación según el tipo de hotel. El City Hotel registró 16.611 reservas, de las cuales 6.933 fueron canceladas, lo que representa un 41,74 %. Por otro lado, el Resort Hotel registró 8.388 reservas, con 2.328 cancelaciones, equivalentes al 27,75 %. Esto representa una diferencia de 13,99 puntos porcentuales entre ambos tipos de hotel. En estos datos, las reservas del City Hotel presentan una mayor proporción de cancelaciones que las del Resort Hotel.**Hallazgo 2 – ADR**     Las reservas canceladas presentan un ADR promedio de 105,04, mientras que las no canceladas registran un promedio de 100,31. La diferencia entre ambos grupos es de aproximadamente 4,72 unidades. También se observa una mediana ligeramente superior en las reservas canceladas (96 frente a 92,79). Sin embargo, las distribuciones presentan una dispersión considerable y valores máximos elevados en ambos grupos. Esto sugiere una diferencia moderada en los valores de ADR según el estado de cancelación, pero no permite afirmar que un ADR más alto provoque una cancelación.**Hallazgo 3 – Lead time**          Las reservas canceladas presentan un lead time promedio de 145,72 días, mientras que las no canceladas tienen un promedio de 79,32 días. La diferencia también se observa en las medianas: 113 días para las canceladas y 44 días para las no canceladas. Además, el primer y tercer cuartil de las reservas canceladas (48 y 218 días) son superiores a los de las no canceladas (8 y 123 días). Estos resultados muestran una asociación entre una mayor anticipación de la reserva y una mayor presencia de cancelaciones. No obstante, no permiten establecer una relación causal.

## 10. Alcances y limitaciones1. **No se puede afirmar causalidad:** las diferencias entre grupos no demuestran que el hotel, la tarifa o la anticipación provoquen una cancelación.2. **No se controlan simultáneamente otras variables:** depósito, segmento, canal, tipo de cliente e historial podrían estar relacionados con los resultados.3. **Los resultados describen este dataset:** no deben generalizarse automáticamente a todos los hoteles, destinos o períodos.4. **Los valores extremos requieren contexto:** las consultas los localizan, pero no permiten clasificarlos automáticamente como errores.Para profundizar será necesario analizar más variables conjuntamente y, en una etapa posterior, evaluar un modelo predictivo.

## 11. Verificación final

In [ ]:
cat(  "Filas en tabla SQL:",  DBI::dbGetQuery(conexion, "SELECT COUNT(*) AS n FROM reservas")$n,  "Columnas en tabla SQL:",  nrow(DBI::dbGetQuery(conexion, "DESCRIBE reservas")),  "Preguntas comparadas: hotel, adr y lead_time.",  "Recorrido utilizado: SQL + R.",  sep = "\n")

## 12. Antes de entregar1. Ejecutar todas las celdas en orden.2. Subir el CSV original y verificar que se cargue correctamente.3. Revisar que las consultas SQL y los gráficos se generen sin errores.4. Completar los tres hallazgos con números reales de las tablas.5. Confirmar que cada interpretación coincida con los resultados.6. Guardar el notebook con las salidas visibles.**Nombre sugerido:** `Aguillon_Fernando_Comision5_Entrega4.ipynb`.

---# ENTREGA 5 — Primer modelo de clasificaciónContenido incorporado desde el notebook original **Aguillon_Fernando_Comision5_Entrega5.ipynb**.

# Trabajo Práctico Integrador – Introducción al Análisis de Datos## Entrega 5: Primer modelo de clasificación (Semana 12)**Apellido y nombre:** Aguillón, Fernando  **Comisión:** 5  **Dataset:** E – reservas hoteleras  **Recorrido:** SQL + R  **Modelo:** Árbol de decisiónEste notebook retoma las decisiones de las Entregas 1 a 4 y la planificación de Semana 11. El objetivo es anticipar si una reserva será cancelada **al momento de registrarla**.**Archivo esperado en Google Colab:** `/content/hotel booking TPI grupo E.csv`Ejecutá todas las celdas en orden, revisá los resultados y completá las interpretaciones. Los resultados de evaluación se calculan con el CSV al ejecutar el notebook; no se inventan valores.

## 1. Objetivo y variables seleccionadasLa variable objetivo es `is_canceled`: `0` = no cancelada y `1` = cancelada. Es un problema de clasificación supervisada porque el modelo aprende con reservas cuyo estado conocemos y luego predice la categoría de otras reservas.Se retoman las ocho variables predictoras de Semana 11:| Variable | Justificación ||---|---|| `hotel` | En la Entrega 4, City Hotel tuvo 41,74 % de cancelaciones y Resort Hotel 27,75 %. || `lead_time` | Las canceladas presentaron una mediana de anticipación de 113 días, frente a 44 días en no canceladas. || `adr` | Se observaron diferencias moderadas entre grupos. Confirmar que el valor esté disponible al registrar la reserva. || `market_segment` | Describe el segmento de mercado. || `distribution_channel` | Identifica el canal por el que llegó la reserva. || `deposit_type` | Describe el tipo de depósito o garantía. || `customer_type` | Caracteriza el tipo de cliente. || `previous_cancellations` | Resume cancelaciones anteriores del cliente. |Las variables categóricas se tratarán como factores de R. Los faltantes categóricos se identificarán como `Sin especificar`; los faltantes numéricos se completarán con la mediana calculada solo en entrenamiento.

## 2. Exclusiones para evitar identificadores y fuga de información- `booking_id`: identificador, no característica predictiva.- `is_canceled`: es la respuesta que queremos predecir, no puede ser predictora.- `reservation_status` y `reservation_status_date`: describen el estado final o su fecha y podrían revelar el resultado.- `assigned_room_type`: habitación finalmente asignada; podría conocerse después del registro.- `booking_changes`: puede acumular modificaciones posteriores.- `days_in_waiting_list`: puede depender de eventos posteriores.- `company`: excluida en la Entrega 2 por su elevada proporción de faltantes.**Consideración sobre ADR:** Se incluye `adr` como variable predictora bajo el supuesto de que su valor estaba disponible al momento de registrar la reserva. Por lo tanto, se considera válida para el objetivo de anticipar cancelaciones. Esta decisión deberá revisarse si se utiliza el modelo en un entorno real, verificando que el valor no haya sido actualizado posteriormente.

## 3. Librerías y entorno SQLDuckDB permite realizar la selección de datos mediante SQL. R se utiliza para preparar los datos, separar entrenamiento y prueba y entrenar el árbol.

In [ ]:
# Configurar identificación del entorno Linuxoptions(HTTPUserAgent = sprintf(  "R/%s R (%s)",  getRversion(),  paste(    getRversion(),    R.version["platform"],    R.version["arch"],    R.version["os"]  )))# Instalar DuckDB desde Positrepositorio <- sprintf(  "https://p3m.dev/cran/latest/bin/linux/manylinux_2_28-%s/%s",  R.version$arch,  paste(R.version$major, R.version$minor, sep = "."))message("Instalando DuckDB desde Posit...")install.packages("duckdb", repos = repositorio)# Verificarmessage("Verificando DuckDB...")library(DBI)library(duckdb)print(packageVersion("duckdb"))message("¡DuckDB está instalado y funcionando!")

## 4. Carga del dataset originalSe carga el CSV original para que esta entrega sea reproducible e independiente de otros notebooks. Si el archivo no está en `/content/`, subilo a Colab antes de continuar.

In [ ]:
ruta_csv <- ruta_csv_compartidaif (!file.exists(ruta_csv)) {  stop("No se encontró el CSV. Subí 'hotel booking TPI grupo E.csv' a /content/ y volvé a ejecutar.")}datos_originales <- read.csv(  ruta_csv, stringsAsFactors = FALSE,  na.strings = c("", "NA"), check.names = FALSE)cat("Filas originales:", nrow(datos_originales), "\n")cat("Columnas originales:", ncol(datos_originales), "\n")stopifnot("is_canceled" %in% names(datos_originales))

## 5. Preparación inicial y continuidad de la Entrega 2Se eliminan duplicados exactos, `booking_id` y `company`; los faltantes de `children` se reemplazan por cero; los faltantes de `agent` y `country` se identifican explícitamente; en `adr` se eliminan solo los valores negativos, conservando ceros y extremos. No se eliminan automáticamente las reservas sin huéspedes.La imputación de las variables del modelo se hará después de separar entrenamiento y prueba.

In [ ]:
df <- datos_originalesfilas_iniciales <- nrow(df)duplicados_exactos <- sum(duplicated(df))faltantes_children <- if ("children" %in% names(df)) sum(is.na(df$children)) else NA_integer_adr_negativos <- if ("adr" %in% names(df)) sum(!is.na(df$adr) & df$adr < 0) else NA_integer_df <- df[!duplicated(df), , drop = FALSE]df <- df[, !(names(df) %in% intersect(c("booking_id", "company"), names(df))), drop = FALSE]if ("children" %in% names(df)) df$children[is.na(df$children)] <- 0if ("agent" %in% names(df)) {  df$agent <- as.character(df$agent)  df$agent[is.na(df$agent) | trimws(df$agent) == ""] <- "Sin agente"}if ("country" %in% names(df)) {  df$country <- as.character(df$country)  df$country[is.na(df$country) | trimws(df$country) == ""] <- "Sin especificar"}if ("adr" %in% names(df)) {  df$adr <- as.numeric(df$adr)  df <- df[is.na(df$adr) | df$adr >= 0, , drop = FALSE]}cat("Duplicados exactos eliminados:", duplicados_exactos, "\n")cat("Faltantes originales de children:", faltantes_children, "\n")cat("ADR negativos detectados:", adr_negativos, "\n")cat("Filas originales:", filas_iniciales, "\n")cat("Filas después de la preparación:", nrow(df), "\n")

## 6. Selección de variables mediante SQLSe seleccionan la variable objetivo y las ocho predictoras. Si falta alguna columna, el notebook se detiene para revisar el nombre real en el CSV, en lugar de sustituirla silenciosamente.

In [ ]:
variables_requeridas <- c(  "is_canceled", "hotel", "lead_time", "adr", "market_segment",  "distribution_channel", "deposit_type", "customer_type",  "previous_cancellations")variables_faltantes <- setdiff(variables_requeridas, names(df))if (length(variables_faltantes) > 0) {  stop(paste("No se encontraron estas columnas:",             paste(variables_faltantes, collapse = ", ")))}df$is_canceled <- as.integer(as.character(df$is_canceled))df$lead_time <- as.numeric(df$lead_time)df$adr <- as.numeric(df$adr)df$previous_cancellations <- as.numeric(df$previous_cancellations)conexion <- DBI::dbConnect(duckdb::duckdb(), dbdir = ":memory:")DBI::dbWriteTable(conexion, "reservas", df, overwrite = TRUE)consulta_modelo <- "SELECT is_canceled, hotel, lead_time, adr, market_segment,       distribution_channel, deposit_type, customer_type,       previous_cancellationsFROM reservasWHERE is_canceled IN (0, 1)"datos_modelo <- DBI::dbGetQuery(conexion, consulta_modelo)cat("Registros seleccionados por SQL:", nrow(datos_modelo), "\n")cat("Columnas seleccionadas:", ncol(datos_modelo), "\n")head(datos_modelo, 10)

## 7. Diagnóstico de faltantes y tiposSe revisan los valores faltantes antes de tratarlos. Todavía no se calculan medianas: primero se separarán los datos en entrenamiento y prueba.

In [ ]:
resumen_faltantes <- data.frame(  variable = names(datos_modelo),  tipo = sapply(datos_modelo, function(x) class(x)[1]),  faltantes = sapply(datos_modelo, function(x) sum(is.na(x))),  porcentaje_faltantes = round(    100 * sapply(datos_modelo, function(x) mean(is.na(x))), 2  ))resumen_faltantes

## 8. Distribución de la variable objetivoSe comprueba cuántos registros pertenecen a cada clase. Si las categorías están desbalanceadas, la exactitud general podría resultar engañosa.

In [ ]:
table(datos_modelo$is_canceled, useNA = "ifany")frecuencia_clases <- as.data.frame(  prop.table(table(datos_modelo$is_canceled)) * 100)names(frecuencia_clases) <- c("is_canceled", "porcentaje")frecuencia_clases

## 9. Separación reproducible: entrenamiento y pruebaSe utiliza una división aproximada de 80 % para entrenamiento y 20 % para prueba, manteniendo la proporción de las clases mediante selección estratificada. La semilla `2026` permite reproducir la separación.El conjunto de prueba queda reservado para evaluar el modelo. No se utilizará para calcular medianas ni parámetros de preparación.

In [ ]:
datos_modelo <- datos_modelo[!is.na(datos_modelo$is_canceled), , drop = FALSE]datos_modelo$is_canceled <- as.integer(datos_modelo$is_canceled)if (!all(datos_modelo$is_canceled %in% c(0L, 1L))) {  stop("is_canceled contiene valores distintos de 0 y 1.")}set.seed(2026)indices_por_clase <- split(seq_len(nrow(datos_modelo)), datos_modelo$is_canceled)indices_entrenamiento <- unlist(lapply(indices_por_clase, function(indices) {  sample(indices, size = floor(0.80 * length(indices)))}), use.names = FALSE)indices_prueba <- setdiff(seq_len(nrow(datos_modelo)), indices_entrenamiento)entrenamiento <- datos_modelo[indices_entrenamiento, , drop = FALSE]prueba <- datos_modelo[indices_prueba, , drop = FALSE]cat("Registros de entrenamiento:", nrow(entrenamiento), "\n")cat("Registros de prueba:", nrow(prueba), "\n")cat("\nProporción de clases en entrenamiento:\n")print(prop.table(table(entrenamiento$is_canceled)))cat("\nProporción de clases en prueba:\n")print(prop.table(table(prueba$is_canceled)))

## 10. Tratamiento de faltantes sin fuga de informaciónVariables numéricas: `lead_time`, `adr` y `previous_cancellations`. Se reemplazan sus faltantes por la mediana calculada exclusivamente en entrenamiento y se aplica esa misma mediana al conjunto de prueba.Variables categóricas: `hotel`, `market_segment`, `distribution_channel`, `deposit_type` y `customer_type`. Los faltantes se identifican como `Sin especificar`. Las categorías que aparezcan solo en prueba se codifican como `Otra categoría`.

In [ ]:
variables_numericas <- c("lead_time", "adr", "previous_cancellations")variables_categoricas <- c(  "hotel", "market_segment", "distribution_channel",  "deposit_type", "customer_type")medianas_entrenamiento <- list()for (variable in variables_numericas) {  entrenamiento[[variable]] <- as.numeric(entrenamiento[[variable]])  prueba[[variable]] <- as.numeric(prueba[[variable]])  mediana_calculada <- median(entrenamiento[[variable]], na.rm = TRUE)  if (!is.finite(mediana_calculada)) {    stop(paste("No se pudo calcular una mediana válida para", variable))  }  medianas_entrenamiento[[variable]] <- mediana_calculada  entrenamiento[[variable]][is.na(entrenamiento[[variable]])] <- mediana_calculada  prueba[[variable]][is.na(prueba[[variable]])] <- mediana_calculada}normalizar_categoria <- function(x) {  x <- trimws(as.character(x))  x[is.na(x) | x == ""] <- "Sin especificar"  x}niveles_entrenamiento <- list()for (variable in variables_categoricas) {  entrenamiento[[variable]] <- normalizar_categoria(entrenamiento[[variable]])  prueba[[variable]] <- normalizar_categoria(prueba[[variable]])  niveles <- unique(c(sort(unique(entrenamiento[[variable]])), "Otra categoría"))  niveles_entrenamiento[[variable]] <- niveles  valores_prueba <- prueba[[variable]]  valores_prueba[!valores_prueba %in% niveles] <- "Otra categoría"  entrenamiento[[variable]] <- factor(entrenamiento[[variable]], levels = niveles)  prueba[[variable]] <- factor(valores_prueba, levels = niveles)}entrenamiento$is_canceled <- factor(  ifelse(entrenamiento$is_canceled == 1, "Cancelada", "No cancelada"),  levels = c("No cancelada", "Cancelada"))prueba$is_canceled <- factor(  ifelse(prueba$is_canceled == 1, "Cancelada", "No cancelada"),  levels = c("No cancelada", "Cancelada"))cat("Medianas calculadas solo con entrenamiento:\n")print(unlist(medianas_entrenamiento))cat("\nFaltantes en entrenamiento después del tratamiento:\n")print(colSums(is.na(entrenamiento)))cat("\nFaltantes en prueba después del tratamiento:\n")print(colSums(is.na(prueba)))

## 11. Entrenamiento del árbol de decisiónSe construye un árbol de decisión con `rpart` y el método `class`, porque la variable objetivo contiene dos categorías. Se limita inicialmente la profundidad a cinco niveles para mantener un árbol más sencillo de inspeccionar. Es un primer modelo, no una configuración optimizada.

In [ ]:
# Comprobar si rpart está instaladoif (!requireNamespace("rpart", quietly = TRUE)) {  message("Instalando rpart...")  install.packages("rpart")}# Cargar el paquetelibrary(rpart)# Comprobar que está disponiblepackageVersion("rpart")message("¡rpart está listo!")

In [ ]:
set.seed(2026)modelo_arbol <- rpart(  is_canceled ~ .,  data = entrenamiento,  method = "class",  control = rpart.control(cp = 0.01, maxdepth = 5, minsplit = 20))print(modelo_arbol)

### Visualización del árbolCada división representa una condición sobre una variable; las hojas muestran la clase asignada por el árbol.

In [ ]:
plot(modelo_arbol, uniform = TRUE,     main = "Árbol de decisión para predecir cancelaciones")text(modelo_arbol, use.n = TRUE, all = TRUE, cex = 0.7)

## 12. Predicciones sobre el conjunto de pruebaEl modelo se aplica a los registros reservados para prueba. Se comparan los estados predichos con los estados reales. La probabilidad estimada es una salida del modelo, no una certeza sobre el futuro.

In [ ]:
predicciones_clase <- predict(modelo_arbol, newdata = prueba, type = "class")predicciones_probabilidad <- predict(modelo_arbol, newdata = prueba, type = "prob")resultados_prueba <- data.frame(  estado_real = prueba$is_canceled,  estado_predicho = predicciones_clase,  probabilidad_cancelacion = round(    predicciones_probabilidad[, "Cancelada"], 4  ))head(resultados_prueba, 20)

## 13. Evaluación inicial: matriz de confusión y métricas- **Verdadero negativo:** no cancelada correctamente identificada.- **Verdadero positivo:** cancelada correctamente identificada.- **Falso positivo:** se predijo cancelación, pero no ocurrió.- **Falso negativo:** se predijo que no se cancelaría, pero sí ocurrió.Se calculan exactitud, sensibilidad, especificidad y precisión para la clase Cancelada.

In [ ]:
matriz_confusion <- table(  Real = prueba$is_canceled,  Predicho = predicciones_clase)matriz_confusionvn <- matriz_confusion["No cancelada", "No cancelada"]fp <- matriz_confusion["No cancelada", "Cancelada"]fn <- matriz_confusion["Cancelada", "No cancelada"]vp <- matriz_confusion["Cancelada", "Cancelada"]exactitud <- sum(diag(matriz_confusion)) / sum(matriz_confusion)sensibilidad <- if ((vp + fn) > 0) vp / (vp + fn) else NA_real_especificidad <- if ((vn + fp) > 0) vn / (vn + fp) else NA_real_precision_cancelacion <- if ((vp + fp) > 0) vp / (vp + fp) else NA_real_metricas <- data.frame(  metrica = c("Exactitud", "Sensibilidad", "Especificidad",              "Precisión de cancelación"),  valor = round(c(exactitud, sensibilidad, especificidad,                  precision_cancelacion), 4))metricas

## 14. Interpretación de las primeras predicciones**1. Primeras predicciones:** De las primeras 20 reservas, el modelo clasificó correctamente 13 y se equivocó en 7. Por ejemplo, en la observación 25, el estado real era cancelada, pero el modelo predijo no cancelada.**2. Utilidad posible:** El modelo podría ayudar al hotel a identificar reservas con mayor probabilidad de cancelación y priorizar su seguimiento, además de contribuir a planificar la disponibilidad de habitaciones.**3. Errores:** Los falsos positivos pueden llevar al hotel a tomar precauciones innecesarias ante reservas que finalmente no se cancelan. Los falsos negativos son especialmente importantes porque representan reservas que el modelo considera seguras, pero que finalmente se cancelan.

## 15. ConclusiónEn esta entrega se desarrolló un modelo de clasificación supervisada mediante un árbol de decisión para anticipar cancelaciones de reservas hoteleras. Se utilizaron ocho variables predictoras seleccionadas mediante SQL y se dividieron los datos en conjuntos de entrenamiento y prueba.El modelo obtuvo una exactitud del 76,72 %, una especificidad del 99,71 % y una precisión de cancelación del 98,73 %. Sin embargo, su sensibilidad fue del 37,67 %, lo que demuestra que no logra identificar una proporción importante de las cancelaciones reales.En la matriz de confusión se observaron 1.155 falsos negativos y solamente 9 falsos positivos. Por lo tanto, considero que el modelo puede ser útil como herramienta de apoyo para reconocer determinados patrones de cancelación, pero todavía presenta limitaciones importantes para anticiparlas. Sería necesario mejorar y comparar el modelo con otras alternativas antes de considerar su aplicación en un hotel real.

---# ENTREGA 6 — Evaluación del modelo de clasificaciónContenido incorporado desde el notebook original **Aguillon_Fernando_Comision5_Entrega6.ipynb**.

# Trabajo Práctico Integrador – Introducción al Análisis de Datos## Entrega 6: Evaluación del modelo de clasificación**Apellido y nombre:** Aguillón, Fernando  **Comisión:** 5  **Dataset:** E – reservas hoteleras  **Recorrido:** SQL + R  **Modelo:** Árbol de decisión (`rpart`)**Propósito:** evaluar el árbol de decisión construido en la Entrega 5 para anticipar cancelaciones de reservas. Se reproducen la selección de variables, la preparación y el entrenamiento; luego se evalúa el modelo sobre observaciones que no utilizó para aprender.**Archivo esperado en Google Colab:** `/content/hotel booking TPI grupo E.csv`

## 1. Objetivo y predictoresLa variable objetivo es `is_canceled`: `0` = no cancelada y `1` = cancelada. Es clasificación supervisada porque el modelo aprende a partir de reservas cuyo resultado conocemos y predice una categoría para otras reservas.Se mantienen los ocho predictores de la Entrega 5:| Variable | Justificación ||---|---|| `hotel` | En el análisis anterior se observaron diferencias en las tasas de cancelación entre City Hotel y Resort Hotel. || `lead_time` | Las reservas canceladas presentaron una mediana de anticipación mayor. || `adr` | Se observaron diferencias entre grupos; debe confirmarse que esté disponible al registrar la reserva. || `market_segment` | Describe el segmento de mercado. || `distribution_channel` | Identifica el canal por el que llegó la reserva. || `deposit_type` | Describe el tipo de depósito o garantía. || `customer_type` | Caracteriza el tipo de cliente. || `previous_cancellations` | Resume cancelaciones anteriores del cliente. |Se excluyen `booking_id` por ser identificador, `company` por su elevada proporción de faltantes y variables como `reservation_status` y `reservation_status_date`, que pueden revelar el resultado final. También se excluyen variables potencialmente posteriores al registro, como `assigned_room_type` y `booking_changes`.**Consideración sobre ADR:** Se conserva esta variable como predictora bajo el supuesto de que su valor estaba disponible al momento de registrar la reserva. Esta decisión permite mantener la continuidad con la Entrega 5. En una aplicación real, sería necesario verificar que el valor utilizado corresponda efectivamente al momento de creación de la reserva y no a una actualización posterior.

## 2. Librerías y entorno SQLDuckDB permite seleccionar los datos mediante SQL. R se utiliza para preparar los datos, entrenar el árbol y evaluar sus predicciones. En Linux se instala DuckDB desde Posit Package Manager sin forzar `type = "binary"`.

In [ ]:
options(repos = c(CRAN = "https://cloud.r-project.org"))options(timeout = 600)if (!requireNamespace("DBI", quietly = TRUE)) install.packages("DBI")if (!requireNamespace("duckdb", quietly = TRUE)) {  arquitectura <- R.version$arch  version_r <- paste(R.version$major, R.version$minor, sep = ".")  repositorio_duckdb <- sprintf(    "https://p3m.dev/cran/latest/bin/linux/manylinux_2_28-%s/%s",    arquitectura, version_r  )  install.packages("duckdb", repos = repositorio_duckdb)}if (!requireNamespace("rpart", quietly = TRUE)) install.packages("rpart")library(DBI)library(duckdb)library(rpart)cat("Paquetes cargados correctamente.\n")cat("DuckDB:", as.character(packageVersion("duckdb")), "\n")cat("rpart:", as.character(packageVersion("rpart")), "\n")

## 3. Carga del dataset originalEl notebook parte del CSV original y no de objetos creados en la Entrega 5. Si el archivo no está disponible en Colab, subilo desde el panel de archivos y verificá que conserve el nombre esperado.

In [ ]:
ruta_csv <- ruta_csv_compartidaif (!file.exists(ruta_csv)) {  stop("No se encontró el CSV. Subí 'hotel booking TPI grupo E.csv' a /content/ y volvé a ejecutar.")}datos_originales <- read.csv(  ruta_csv, stringsAsFactors = FALSE,  na.strings = c("", "NA"), check.names = FALSE)cat("Filas originales:", nrow(datos_originales), "\n")cat("Columnas originales:", ncol(datos_originales), "\n")stopifnot("is_canceled" %in% names(datos_originales))

## 4. Preparación inicialSe mantienen los criterios anteriores: eliminar duplicados exactos, excluir `booking_id` y `company`, reemplazar faltantes de `children` por cero, identificar faltantes de `agent` y `country` mediante categorías explícitas y eliminar solo valores negativos de `adr`. Se conservan ceros y extremos de ADR. No se eliminan automáticamente reservas sin huéspedes.La imputación de los predictores se realiza después de separar entrenamiento y prueba.

In [ ]:
df <- datos_originalesfilas_iniciales <- nrow(df)duplicados_exactos <- sum(duplicated(df))faltantes_children <- if ("children" %in% names(df)) sum(is.na(df$children)) else NA_integer_adr_negativos <- if ("adr" %in% names(df)) sum(!is.na(df$adr) & df$adr < 0) else NA_integer_df <- df[!duplicated(df), , drop = FALSE]df <- df[, !(names(df) %in% intersect(c("booking_id", "company"), names(df))), drop = FALSE]if ("children" %in% names(df)) df$children[is.na(df$children)] <- 0if ("agent" %in% names(df)) {  df$agent <- as.character(df$agent)  df$agent[is.na(df$agent) | trimws(df$agent) == ""] <- "Sin agente"}if ("country" %in% names(df)) {  df$country <- as.character(df$country)  df$country[is.na(df$country) | trimws(df$country) == ""] <- "Sin especificar"}if ("adr" %in% names(df)) {  df$adr <- as.numeric(df$adr)  df <- df[is.na(df$adr) | df$adr >= 0, , drop = FALSE]}cat("Duplicados exactos eliminados:", duplicados_exactos, "\n")cat("Faltantes originales de children:", faltantes_children, "\n")cat("ADR negativos detectados:", adr_negativos, "\n")cat("Filas originales:", filas_iniciales, "\n")cat("Filas después de preparar:", nrow(df), "\n")

## 5. Selección de variables mediante SQLSe seleccionan la variable objetivo y los mismos ocho predictores de la Entrega 5.

In [ ]:
variables_requeridas <- c(  "is_canceled", "hotel", "lead_time", "adr", "market_segment",  "distribution_channel", "deposit_type", "customer_type",  "previous_cancellations")variables_faltantes <- setdiff(variables_requeridas, names(df))if (length(variables_faltantes) > 0) {  stop(paste("No se encontraron:", paste(variables_faltantes, collapse = ", ")))}df$is_canceled <- as.integer(as.character(df$is_canceled))df$lead_time <- as.numeric(df$lead_time)df$adr <- as.numeric(df$adr)df$previous_cancellations <- as.numeric(df$previous_cancellations)conexion <- DBI::dbConnect(duckdb::duckdb(), dbdir = ":memory:")DBI::dbWriteTable(conexion, "reservas", df, overwrite = TRUE)consulta_modelo <- "SELECT is_canceled, hotel, lead_time, adr, market_segment,       distribution_channel, deposit_type, customer_type,       previous_cancellationsFROM reservasWHERE is_canceled IN (0, 1)"datos_modelo <- DBI::dbGetQuery(conexion, consulta_modelo)cat("Registros seleccionados por SQL:", nrow(datos_modelo), "\n")cat("Columnas seleccionadas:", ncol(datos_modelo), "\n")head(datos_modelo, 10)

## 6. Diagnóstico de faltantes y clasesSe revisan los faltantes y la distribución de la variable objetivo. Si las clases están desbalanceadas, la exactitud por sí sola puede resultar engañosa.

In [ ]:
resumen_faltantes <- data.frame(  variable = names(datos_modelo),  tipo = sapply(datos_modelo, function(x) class(x)[1]),  faltantes = sapply(datos_modelo, function(x) sum(is.na(x))),  porcentaje_faltantes = round(    100 * sapply(datos_modelo, function(x) mean(is.na(x))), 2  ))resumen_faltantes

In [ ]:
table(datos_modelo$is_canceled, useNA = "ifany")frecuencia_clases <- as.data.frame(prop.table(table(datos_modelo$is_canceled)) * 100)names(frecuencia_clases) <- c("is_canceled", "porcentaje")frecuencia_clases

## 7. Separación entre entrenamiento y pruebaSe utiliza aproximadamente el **80 %** para entrenamiento y el **20 %** para prueba, manteniendo la proporción de clases mediante selección estratificada. La semilla `2026` permite reproducir la separación.El conjunto de prueba contiene observaciones que el modelo no utilizó para aprender. Evaluarlo allí permite aproximarse a su capacidad de **generalización**. Si el rendimiento en prueba empeorara mucho respecto del entrenamiento, podría ser una señal de **sobreajuste**.

In [ ]:
datos_modelo <- datos_modelo[!is.na(datos_modelo$is_canceled), , drop = FALSE]datos_modelo$is_canceled <- as.integer(datos_modelo$is_canceled)if (!all(datos_modelo$is_canceled %in% c(0L, 1L))) {  stop("is_canceled contiene valores distintos de 0 y 1.")}set.seed(2026)indices_por_clase <- split(seq_len(nrow(datos_modelo)), datos_modelo$is_canceled)indices_entrenamiento <- unlist(lapply(indices_por_clase, function(indices) {  sample(indices, size = floor(0.80 * length(indices)))}), use.names = FALSE)indices_prueba <- setdiff(seq_len(nrow(datos_modelo)), indices_entrenamiento)entrenamiento <- datos_modelo[indices_entrenamiento, , drop = FALSE]prueba <- datos_modelo[indices_prueba, , drop = FALSE]cat("Registros de entrenamiento:", nrow(entrenamiento), "\n")cat("Registros de prueba:", nrow(prueba), "\n")cat("\nProporción de clases en entrenamiento:\n")print(prop.table(table(entrenamiento$is_canceled)))cat("\nProporción de clases en prueba:\n")print(prop.table(table(prueba$is_canceled)))

## 8. Tratamiento de faltantes sin fuga de informaciónSe conserva el procedimiento de la Entrega 5:- Variables numéricas (`lead_time`, `adr`, `previous_cancellations`): faltantes reemplazados por la mediana calculada exclusivamente en entrenamiento. Esa misma mediana se aplica a prueba.- Variables categóricas (`hotel`, `market_segment`, `distribution_channel`, `deposit_type`, `customer_type`): faltantes identificados como `Sin especificar`. Las categorías que aparezcan solo en prueba se codifican como `Otra categoría`.Así, el conjunto de prueba no se utiliza para aprender los valores de preparación.

In [ ]:
variables_numericas <- c("lead_time", "adr", "previous_cancellations")variables_categoricas <- c(  "hotel", "market_segment", "distribution_channel",  "deposit_type", "customer_type")medianas_entrenamiento <- list()for (variable in variables_numericas) {  entrenamiento[[variable]] <- as.numeric(entrenamiento[[variable]])  prueba[[variable]] <- as.numeric(prueba[[variable]])  mediana_calculada <- median(entrenamiento[[variable]], na.rm = TRUE)  if (!is.finite(mediana_calculada)) {    stop(paste("No se pudo calcular una mediana válida para", variable))  }  medianas_entrenamiento[[variable]] <- mediana_calculada  entrenamiento[[variable]][is.na(entrenamiento[[variable]])] <- mediana_calculada  prueba[[variable]][is.na(prueba[[variable]])] <- mediana_calculada}normalizar_categoria <- function(x) {  x <- trimws(as.character(x))  x[is.na(x) | x == ""] <- "Sin especificar"  x}for (variable in variables_categoricas) {  entrenamiento[[variable]] <- normalizar_categoria(entrenamiento[[variable]])  prueba[[variable]] <- normalizar_categoria(prueba[[variable]])  niveles <- unique(c(sort(unique(entrenamiento[[variable]])), "Otra categoría"))  valores_prueba <- prueba[[variable]]  valores_prueba[!valores_prueba %in% niveles] <- "Otra categoría"  entrenamiento[[variable]] <- factor(entrenamiento[[variable]], levels = niveles)  prueba[[variable]] <- factor(valores_prueba, levels = niveles)}entrenamiento$is_canceled <- factor(  ifelse(entrenamiento$is_canceled == 1, "Cancelada", "No cancelada"),  levels = c("No cancelada", "Cancelada"))prueba$is_canceled <- factor(  ifelse(prueba$is_canceled == 1, "Cancelada", "No cancelada"),  levels = c("No cancelada", "Cancelada"))cat("Medianas calculadas solo con entrenamiento:\n")print(unlist(medianas_entrenamiento))cat("\nFaltantes en entrenamiento:\n")print(colSums(is.na(entrenamiento)))cat("\nFaltantes en prueba:\n")print(colSums(is.na(prueba)))

## 9. Entrenamiento del árbol de decisiónSe reproduce el mismo modelo y configuración inicial de la Entrega 5: `rpart`, método `class`, `cp = 0.01`, profundidad máxima 5 y `minsplit = 20`. No se incorporan otros algoritmos ni se optimizan hiperparámetros: el propósito es evaluar el modelo trabajado.

In [ ]:
set.seed(2026)modelo_arbol <- rpart(  is_canceled ~ .,  data = entrenamiento,  method = "class",  control = rpart.control(cp = 0.01, maxdepth = 5, minsplit = 20))print(modelo_arbol)

In [ ]:
plot(modelo_arbol, uniform = TRUE,     main = "Árbol de decisión para anticipar cancelaciones")text(modelo_arbol, use.n = TRUE, all = TRUE, cex = 0.7)

## 10. Predicciones sobre el conjunto de pruebaSe predice la clase de cada reserva del conjunto de prueba y se compara el estado real con el predicho. La probabilidad estimada es una salida del modelo, no una certeza.

In [ ]:
predicciones_clase <- predict(modelo_arbol, newdata = prueba, type = "class")predicciones_probabilidad <- predict(modelo_arbol, newdata = prueba, type = "prob")resultados_prueba <- data.frame(  estado_real = prueba$is_canceled,  estado_predicho = predicciones_clase,  probabilidad_cancelacion = round(    predicciones_probabilidad[, "Cancelada"], 4  ))head(resultados_prueba, 20)

## 11. Matriz de confusiónConsideramos **positiva** la clase `Cancelada`.| Resultado | Significado ||---|---|| **Verdadero positivo (VP)** | La reserva se canceló y el modelo predijo cancelación. || **Verdadero negativo (VN)** | La reserva no se canceló y el modelo predijo que no se cancelaría. || **Falso positivo (FP)** | El modelo predijo cancelación, pero la reserva finalmente no se canceló. || **Falso negativo (FN)** | El modelo predijo que no se cancelaría, pero finalmente sí se canceló. |En la matriz de R, las filas son el estado real y las columnas son el estado predicho.

In [ ]:
matriz_confusion <- table(  Real = prueba$is_canceled,  Predicho = predicciones_clase)matriz_confusionvn <- as.numeric(matriz_confusion["No cancelada", "No cancelada"])fp <- as.numeric(matriz_confusion["No cancelada", "Cancelada"])fn <- as.numeric(matriz_confusion["Cancelada", "No cancelada"])vp <- as.numeric(matriz_confusion["Cancelada", "Cancelada"])conteos <- data.frame(  resultado = c("Verdaderos positivos (VP)", "Verdaderos negativos (VN)",                "Falsos positivos (FP)", "Falsos negativos (FN)"),  cantidad = c(vp, vn, fp, fn))conteos

## 12. Métricas de clasificaciónLas métricas se calculan considerando `Cancelada` como clase positiva:- **Accuracy (exactitud):** proporción total de reservas clasificadas correctamente.- **Precision (precisión):** de las reservas que el modelo predijo como canceladas, qué proporción realmente se canceló.- **Recall (sensibilidad):** de todas las reservas que realmente se cancelaron, qué proporción detectó el modelo.- **F1-score:** media armónica entre precision y recall; resume el equilibrio entre ambas.\[Accuracy = \frac{VP + VN}{VP + VN + FP + FN}\]\[Precision = \frac{VP}{VP + FP}\qquadRecall = \frac{VP}{VP + FN}\]\[F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}\]Si un denominador es cero, la métrica se informa como `NA`.

In [ ]:
total <- vp + vn + fp + fnaccuracy <- if (total > 0) (vp + vn) / total else NA_real_precision <- if ((vp + fp) > 0) vp / (vp + fp) else NA_real_recall <- if ((vp + fn) > 0) vp / (vp + fn) else NA_real_f1_score <- if (is.finite(precision) && is.finite(recall) &&                (precision + recall) > 0) {  2 * precision * recall / (precision + recall)} else {  NA_real_}metricas <- data.frame(  metrica = c("Accuracy", "Precision", "Recall", "F1-score"),  valor = round(c(accuracy, precision, recall, f1_score), 4),  porcentaje = round(100 * c(accuracy, precision, recall, f1_score), 2))metricas

### Interpretación de las métricas obtenidas- Un **accuracy** alto indica que el modelo acierta en una proporción elevada del total, pero puede ocultar dificultades para detectar cancelaciones si las clases están desbalanceadas.- Una **precision** alta significa que, cuando el modelo alerta sobre una cancelación, suele acertar. Una precision baja implica más falsas alarmas.- Un **recall** alto significa que el modelo detecta una proporción elevada de las cancelaciones reales. Un recall bajo implica que deja escapar muchas cancelaciones.- Un **F1-score** alto indica un mejor equilibrio entre precision y recall.

In [ ]:
cat(sprintf("Accuracy: %.4f. Clasificó correctamente el %.2f%% del conjunto de prueba.\n",            accuracy, 100 * accuracy))if (is.finite(precision)) {  cat(sprintf("Precision: %.4f. De las reservas marcadas como canceladas, el %.2f%% realmente se canceló.\n",              precision, 100 * precision))} else {  cat("Precision: no se calcula porque el modelo no predijo casos positivos.\n")}if (is.finite(recall)) {  cat(sprintf("Recall: %.4f. Detectó el %.2f%% de las cancelaciones reales.\n",              recall, 100 * recall))} else {  cat("Recall: no se calcula porque no hay cancelaciones reales en prueba.\n")}if (is.finite(f1_score)) {  cat(sprintf("F1-score: %.4f. Resume el equilibrio entre precision y recall.\n", f1_score))} else {  cat("F1-score: no se pudo calcular con los valores disponibles.\n")}

## 13. Interpretación de los errores**Falso positivo (FP):** el modelo alerta que una reserva se cancelará, pero el huésped finalmente sí se presenta. Puede provocar seguimiento innecesario, contactos que incomoden al cliente o decisiones equivocadas sobre la disponibilidad.**Falso negativo (FN):** el modelo predice que una reserva se mantendrá, pero finalmente se cancela. El hotel podría perder la oportunidad de anticiparse, contactar al cliente, ajustar la ocupación o volver a ofrecer esa habitación con tiempo.Un falso negativo puede ser especialmente relevante para la planificación porque una cancelación no anticipada puede dejar capacidad disponible sin tiempo suficiente para recuperarla. Sin embargo, la importancia relativa depende de los costos y la estrategia del establecimiento: demasiados falsos positivos también generan acciones innecesarias o perjudican la experiencia del huésped.En los resultados obtenidos se observa una diferencia importante entre ambos tipos de errores: hubo 1.155 falsos negativos frente a solamente 9 falsos positivos. Esto indica que el modelo tiene dificultades para detectar reservas que finalmente se cancelan. Para un hotel, los falsos negativos pueden ser especialmente problemáticos porque reducen las posibilidades de anticipar la cancelación y volver a ofrecer la habitación disponible.

In [ ]:
cat("Falsos positivos:", fp, "\n")cat("Falsos negativos:", fn, "\n")if (fp > fn) {  cat("En el conjunto de prueba hubo más falsos positivos que falsos negativos.\n")} else if (fn > fp) {  cat("En el conjunto de prueba hubo más falsos negativos que falsos positivos.\n")} else {  cat("Hubo la misma cantidad de falsos positivos y falsos negativos.\n")}

## 14. ConclusiónEl árbol de decisión aporta información limitada para anticipar cancelaciones. Obtuvo un accuracy del 76,72 %, una precision del 98,73 %, un recall del 37,67 % y un F1-score del 54,53 %.Aunque presenta una precisión elevada, solamente detectó el 37,67 % de las cancelaciones reales. En la matriz de confusión se observaron 9 falsos positivos y 1.155 falsos negativos. Esto significa que el modelo casi no genera falsas alarmas, pero deja pasar una cantidad importante de cancelaciones.Considero que puede servir como una primera herramienta de orientación para identificar patrones de cancelación, pero todavía no es suficientemente eficaz para anticiparlas por sí solo. Entre sus limitaciones se encuentran la utilización de un único árbol de decisión con parámetros iniciales, la evaluación sobre una partición del dataset y la posibilidad de que los patrones no se mantengan en otros hoteles o períodos. Además, se trabajó bajo el supuesto de que los predictores, incluido ADR, estaban disponibles al registrar la reserva.

In [ ]:
cat("RESUMEN DE EVALUACIÓN\n")cat("---------------------\n")cat("Reservas evaluadas:", total, "\n")cat("Verdaderos positivos:", vp, "\n")cat("Verdaderos negativos:", vn, "\n")cat("Falsos positivos:", fp, "\n")cat("Falsos negativos:", fn, "\n")cat("Accuracy:", round(accuracy, 4), "\n")cat("Precision:", round(precision, 4), "\n")cat("Recall:", round(recall, 4), "\n")cat("F1-score:", round(f1_score, 4), "\n")